In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

# ── Messidor-2 dataset path (as supplied) ─────────────────────────────
mpath        = '/content/drive/MyDrive//'
image_folder = mpath + 'dataset2-esrgan//training//Images'

# ── Derived working paths ─────────────────────────────────────────────
metrics_dir      = mpath + 'models_metrics_messidor2'
results_dir      = mpath + 'Results_messidor2'
ckpt_dir         = mpath + 'model_messidor2'
plots_dir        = os.path.join(results_dir, 'plots')

# Path to the ORIGINAL (non-Messidor2) trained proposed-model checkpoint,
# used for the zero-shot cross-dataset generalization test.
# This is the checkpoint produced by the original DHA-ESRGAN notebook.
original_ckpt_path = os.path.join(mpath, 'model', '_ESRGAN.pth')

for d in [image_folder, metrics_dir, results_dir, ckpt_dir, plots_dir]:
    os.makedirs(d, exist_ok=True)

print('Messidor-2 image folder :', image_folder)
print('Metrics dir             :', metrics_dir)
print('Results dir              :', results_dir)
print('Checkpoint dir           :', ckpt_dir)
print('Original (source-domain) checkpoint expected at:', original_ckpt_path)


In [ ]:
# ── Core imports used throughout the notebook ─────────────────────────
try:
    from einops import rearrange
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'einops', '-q'])
    from einops import rearrange

import io, os, csv, math, time, glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19
from torchvision import transforms
from torchvision.ops import DeformConv2d
from PIL import Image

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from skimage.filters import sobel
from torch.nn.functional import mse_loss, l1_loss

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
import matplotlib.pyplot as plt
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import DeformConv2d
from PIL import Image
import glob
import torch.nn.utils as spectral_norm_utils # Import spectral_norm_utils
import functools # Import functools for partial

# ----------------------
# Deformable Convolution Block with Batch Normalization
# ----------------------
class DeformableConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(DeformableConvBlock, self).__init__()
        self.offset_conv = nn.Conv2d(in_channels, 2 * kernel_size * kernel_size, kernel_size=kernel_size, padding=padding)
        self.dcn = DeformConv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        offset = self.offset_conv(x)
        x = self.dcn(x, offset)
        return self.relu(self.batch_norm(x))

# ----------------------
# Attention Block (Channel & Spatial Attention)
# ----------------------
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super(AttentionBlock, self).__init__()
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.spatial_att = nn.Sequential(
            nn.Conv2d(channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.channel_att(x) * self.spatial_att(x)

def make_layer(block, n_layers):
    layers = []
    for _ in range(n_layers):
        layers.append(block())
    return nn.Sequential(*layers)


class ResidualDenseBlock_5C(nn.Module):
    def __init__(self, nf=64, gc=32, bias=True):
        super(ResidualDenseBlock_5C, self).__init__()
        # gc: growth channel, i.e. intermediate channels
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1, bias=bias)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1, bias=bias)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1, bias=bias)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1, bias=bias)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1, bias=bias)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x


class RRDB(nn.Module):
    '''Residual in Residual Dense Block'''

    def __init__(self, nf, gc=32):
        super(RRDB, self).__init__()
        self.RDB1 = ResidualDenseBlock_5C(nf, gc)
        self.RDB2 = ResidualDenseBlock_5C(nf, gc)
        self.RDB3 = ResidualDenseBlock_5C(nf, gc)

    def forward(self, x):
        out = self.RDB1(x)
        out = self.RDB2(out)
        out = self.RDB3(out)
        return out * 0.2 + x

# ----------------------
# Generator with Batch Normalization
# ----------------------
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_blocks=8, upscale_factor=4):
        super(Generator, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.prelu = nn.LeakyReLU(0.2, inplace=True)

        # Residual Blocks with RRDB
        RRDB_block_f = functools.partial(RRDB, nf=64, gc=32)
        self.RRDB_trunk = make_layer(RRDB_block_f, num_blocks)
        self.trunk_conv = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=True)

        # Upsampling with Spectral Normalization
        self.upconv1 = spectral_norm_utils.spectral_norm(nn.Conv2d(64, 256, kernel_size=3, padding=1))
        self.upconv2 = spectral_norm_utils.spectral_norm(nn.Conv2d(64, 256, kernel_size=3, padding=1))
        self.pixel_shuffle = nn.PixelShuffle(2)  # 2x upsampling

        # New layers for upsampling stage as per user request
        self.conv_after_upsample = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.spectral_conv_final = spectral_norm_utils.spectral_norm(nn.Conv2d(64, 64, kernel_size=3, padding=1))
        self.final_output_block = nn.Sequential(
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, x):
        residual_global = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False)  # Resize LR to match SR

        # Initial feature extraction
        initial_features = self.prelu(self.bn1(self.conv1(x)))

        # RRDB trunk processing
        trunk_features = self.RRDB_trunk(initial_features)
        # Global residual connection for the RRDB trunk, followed by trunk_conv
        x = initial_features + self.trunk_conv(trunk_features)

        # Upsampling stage
        x = self.pixel_shuffle(self.upconv1(x))
        x = self.pixel_shuffle(self.upconv2(x))

        # Final layers
        x = self.conv_after_upsample(x)
        x = self.spectral_conv_final(x)
        x = self.final_output_block(x)

        return x + residual_global  # Add the global residual connection


# ----------------------
# Discriminator with Batch Normalization
# ----------------------
class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # First convolutional layer: Output 32 feature maps (channels)
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # Second convolutional layer: Output 64 feature maps (channels)
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # Third convolutional layer: Output 128 feature maps (channels)
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # Fourth convolutional layer: Output 256 feature maps (channels)
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 1), # Adjusted input features for Linear layer
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# ----------------------
# Perceptual Loss with Pretrained VGG19
# ----------------------
class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = vgg19(pretrained=True).features[:18]
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.loss = nn.MSELoss()

    def forward(self, x, y):
        return self.loss(self.vgg(x), self.vgg(y))

# ----------------------
# Dataset
# ----------------------
class SuperResolutionDataset(Dataset):
    def __init__(self, image_dir, scale_factor=4, fixed_size=(256, 256)):
        self.image_paths = [os.path.join(image_dir, file) for file in os.listdir(image_dir) if file.endswith(('jpg', 'jpeg', 'png'))]
        self.scale_factor = scale_factor
        self.fixed_size = fixed_size
        self.hr_transform = transforms.Compose([
            transforms.Resize(self.fixed_size, interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        hr_img = self.hr_transform(img)
        lr_size = (self.fixed_size[0] // self.scale_factor, self.fixed_size[1] // self.scale_factor)
        lr_transform = transforms.Compose([
            transforms.Resize(lr_size, interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])
        lr_img = lr_transform(img)
        return lr_img, hr_img


# ----------------------
# Training
# ----------------------
def train_ESRGAN(generator, discriminator, perceptual_loss, dataloader, num_epochs=50, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator, discriminator, perceptual_loss = generator.to(device), discriminator.to(device), perceptual_loss.to(device)

    optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion_gan = nn.BCELoss()

    for epoch in range(num_epochs):
        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            sr_images = generator(lr_images)

            optimizer_D.zero_grad()
            real_labels = torch.ones(lr_images.size(0), 1, device=device)
            fake_labels = torch.zeros(lr_images.size(0), 1, device=device)
            loss_d = (criterion_gan(discriminator(hr_images), real_labels) + criterion_gan(discriminator(sr_images.detach()), fake_labels)) / 2
            loss_d.backward()
            optimizer_D.step()

            optimizer_G.zero_grad()
            loss_g = perceptual_loss(sr_images, hr_images) + 1e-3 * criterion_gan(discriminator(sr_images), real_labels)
            loss_g.backward()
            optimizer_G.step()

        print(f"Epoch [{epoch + 1}/{num_epochs}] - Loss D: {loss_d.item():.4f} - Loss G: {loss_g.item():.4f}")

    return generator


LR Shape: torch.Size([3, 64, 64]), HR Shape: torch.Size([3, 256, 256])


In [ ]:
# ----------------------------------------------------------------------
# VDSR  (unchanged from original notebook)
# ----------------------------------------------------------------------
class VDSR(nn.Module):
    def __init__(self):
        super(VDSR, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.residual_layers = nn.Sequential(*[nn.Conv2d(8, 8, kernel_size=3, padding=1) for _ in range(3)])
        self.conv_final = nn.Conv2d(8, 3, kernel_size=3, padding=1)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.relu(residual)
        residual = self.residual_layers(residual)
        residual = self.conv_final(residual)
        return x + residual  # Residual learning


# ----------------------------------------------------------------------
# EDSR  (unchanged from original notebook)
# ----------------------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels, res_scale=0.1):
        super(ResidualBlock, self).__init__()
        self.res_scale = res_scale
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.relu(residual)
        residual = self.conv2(residual)
        return x + residual * self.res_scale


class EDSR(nn.Module):
    def __init__(self, num_channels=3, num_blocks=16, feature_size=64, res_scale=0.1):
        super(EDSR, self).__init__()
        self.conv1 = nn.Conv2d(num_channels, feature_size, kernel_size=3, padding=1)
        self.residual_layers = nn.Sequential(*[ResidualBlock(feature_size, res_scale) for _ in range(num_blocks)])
        self.conv2 = nn.Conv2d(feature_size, num_channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.residual_layers(residual)
        residual = self.conv2(residual)
        return residual


# ----------------------------------------------------------------------
# RCAN  (unchanged from original notebook)
# ----------------------------------------------------------------------
class CALayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(CALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv1 = nn.Conv2d(channel, channel // reduction, 1, padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channel // reduction, channel, 1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.avg_pool(x)
        avg_out = self.conv1(avg_out)
        avg_out = self.relu(avg_out)
        avg_out = self.conv2(avg_out)
        avg_out = self.sigmoid(avg_out)
        return x * avg_out


class RCAB(nn.Module):
    def __init__(self, channels):
        super(RCAB, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.ca = CALayer(channels)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.relu(residual)
        residual = self.conv2(residual)
        residual = self.ca(residual)
        return x + residual


class ResidualGroup(nn.Module):
    def __init__(self, num_blocks, channels):
        super(ResidualGroup, self).__init__()
        self.rcab_layers = nn.Sequential(*[RCAB(channels) for _ in range(num_blocks)])
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        return x + self.conv(self.rcab_layers(x))


class RCAN(nn.Module):
    def __init__(self, num_channels=3, num_res_groups=10, num_res_blocks=20, feature_size=64):
        super(RCAN, self).__init__()
        self.conv1 = nn.Conv2d(num_channels, feature_size, kernel_size=3, padding=1)
        self.residual_groups = nn.Sequential(*[ResidualGroup(num_res_blocks, feature_size) for _ in range(num_res_groups)])
        self.conv2 = nn.Conv2d(feature_size, num_channels, kernel_size=3, padding=1)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.residual_groups(residual)
        residual = self.conv2(residual)
        return x + residual

print('VDSR, EDSR, RCAN defined (unchanged from original notebook).')


VDSR, EDSR, RCAN defined (unchanged from original notebook).


In [ ]:
# ----------------------------------------------------------------------
# Swin2SR
# ----------------------------------------------------------------------
def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return windows.view(-1, window_size, window_size, C)

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        self.register_buffer('relative_position_index', relative_coords.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        rel_pos_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size * self.window_size, self.window_size * self.window_size, -1)
        attn = attn + rel_pos_bias.permute(2, 0, 1).contiguous().unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.attn_drop(self.softmax(attn))
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))


class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size=8, shift_size=0, mlp_ratio=4.):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, dim))

    def forward(self, x, attn_mask=None):
        B, H, W, C = x.shape
        shortcut = x
        x = self.norm1(x)

        pad_b = (self.window_size - H % self.window_size) % self.window_size
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        x = F.pad(x, (0, 0, 0, pad_r, 0, pad_b))
        _, Hp, Wp, _ = x.shape

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        x_windows = window_partition(x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows, mask=attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_windows, self.window_size, Hp, Wp)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = x[:, :H, :W, :].contiguous()
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class Swin2SRPatchEmbed(nn.Module):
    def __init__(self, in_channels=3, embed_dim=64):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=3, padding=1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x), H, W


class PatchUnembed(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, x, H, W):
        return x.transpose(1, 2).view(-1, self.embed_dim, H, W)


class RSTB(nn.Module):
    def __init__(self, dim, num_heads, num_blocks=6, window_size=8, mlp_ratio=4.):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2, mlp_ratio=mlp_ratio
            )
            for i in range(num_blocks)
        ])
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)
        self.patch_unembed = PatchUnembed(embed_dim=dim)

    def forward(self, x, attn_mask=None):
        residual = x
        B, C, H, W = x.shape
        seq = x.flatten(2).transpose(1, 2).view(B, H, W, C)
        for blk in self.blocks:
            seq = blk(seq, attn_mask)
        x = self.patch_unembed(seq.view(B, H * W, C), H, W)
        return self.conv(x) + residual


class Swin2SR(nn.Module):
    '''SOTA Swin2SR Super-Resolution Network. Paper: Conde et al., ECCV Workshop 2022 | arXiv: 2209.11345'''
    def __init__(self, in_channels=3, out_channels=3, embed_dim=64, num_heads=4,
                 num_rstb=4, num_swin_blocks=4, window_size=8, mlp_ratio=4., scale=4):
        super().__init__()
        self.window_size = window_size
        self.scale = scale

        self.shallow_feat = nn.Conv2d(in_channels, embed_dim, 3, padding=1)
        self.rstb_layers = nn.ModuleList([
            RSTB(dim=embed_dim, num_heads=num_heads, num_blocks=num_swin_blocks,
                 window_size=window_size, mlp_ratio=mlp_ratio)
            for _ in range(num_rstb)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, padding=1)
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(embed_dim, embed_dim * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv_last = nn.Conv2d(embed_dim, out_channels, 3, padding=1)

    def check_image_size(self, x):
        _, _, h, w = x.size()
        mod_pad_h = (self.window_size - h % self.window_size) % self.window_size
        mod_pad_w = (self.window_size - w % self.window_size) % self.window_size
        return F.pad(x, (0, mod_pad_w, 0, mod_pad_h), 'reflect')

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]
        x = self.check_image_size(x)
        bicubic = F.interpolate(x, scale_factor=self.scale, mode='bicubic', align_corners=False)
        feat = self.shallow_feat(x)
        deep = feat
        for rstb in self.rstb_layers:
            deep = rstb(deep)
        B, C, Hp, Wp = deep.shape
        deep = self.norm(deep.flatten(2).transpose(1, 2))
        deep = deep.transpose(1, 2).view(B, C, Hp, Wp)
        deep = self.conv_after_body(deep) + feat
        out = self.conv_last(self.upsample(deep))
        out = out[:, :, :H * self.scale, :W * self.scale]
        bic = bicubic[:, :, :H * self.scale, :W * self.scale]
        return out + bic

print('Swin2SR (RSTB-based) defined (unchanged from original notebook).')


In [ ]:
# ----------------------------------------------------------------------
# VisionMambaSR
# ----------------------------------------------------------------------
class SSMBlock(nn.Module):
    def __init__(self, dim, expand=2):
        super().__init__()
        inner = int(dim * expand)
        self.norm = nn.LayerNorm(dim)
        self.in_proj = nn.Linear(dim, inner * 2)
        self.conv1d = nn.Conv1d(inner, inner, 3, padding=1, groups=inner)
        self.act = nn.SiLU()
        self.out_proj = nn.Linear(inner, dim)
        self.D = nn.Parameter(torch.ones(inner))

    def forward(self, x):
        B, C, H, W = x.shape
        res = x
        x = self.norm(rearrange(x, 'b c h w -> b (h w) c'))
        xv, z = self.in_proj(x).chunk(2, dim=-1)
        xv = rearrange(self.conv1d(rearrange(xv, 'b l c -> b c l')), 'b c l -> b l c')
        xv = self.act(xv) * torch.sigmoid(z)
        xv = xv * self.D
        out = self.out_proj(xv)
        return rearrange(out, 'b (h w) c -> b c h w', h=H, w=W) + res


class VSSLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ssm = SSMBlock(dim)
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))

    def forward(self, x):
        x = self.ssm(x)
        B, C, H, W = x.shape
        xf = self.ff(self.norm(rearrange(x, 'b c h w -> b (h w) c')))
        return x + rearrange(xf, 'b (h w) c -> b c h w', h=H, w=W)


class VisionMambaSR(nn.Module):
    '''MambaIR-inspired SR model (Guo et al., arXiv 2402.15648)'''
    def __init__(self, in_channels=3, out_channels=3, dim=64, n_layers=6, scale=4):
        super().__init__()
        self.entry = nn.Conv2d(in_channels, dim, 3, padding=1)
        self.body = nn.Sequential(*[VSSLayer(dim) for _ in range(n_layers)])
        self.tail = nn.Sequential(
            nn.Conv2d(dim, dim * scale * scale, 3, padding=1),
            nn.PixelShuffle(scale),
            nn.Conv2d(dim, out_channels, 3, padding=1))
        self.scale = scale

    def forward(self, x):
        bic = F.interpolate(x, scale_factor=self.scale, mode='bicubic', align_corners=False)
        f = self.entry(x)
        f = self.body(f)
        return self.tail(f) + bic

print('VisionMambaSR defined (unchanged from original notebook).')


In [ ]:
# ----------------------------------------------------------------------
# DiffusionSR
# ----------------------------------------------------------------------
class SinusoidalPE(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        emb = t[:, None].float() * freq[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class DiffResBlock(nn.Module):
    def __init__(self, ic, oc, td):
        super().__init__()
        self.n1 = nn.GroupNorm(2, ic)
        self.c1 = nn.Conv2d(ic, oc, 3, padding=1)
        self.te = nn.Linear(td, oc)
        self.n2 = nn.GroupNorm(2, oc)
        self.c2 = nn.Conv2d(oc, oc, 3, padding=1)
        self.sk = nn.Conv2d(ic, oc, 1) if ic != oc else nn.Identity()

    def forward(self, x, te):
        h = self.c1(F.silu(self.n1(x))) + self.te(F.silu(te))[:, :, None, None]
        return self.c2(F.silu(self.n2(h))) + self.sk(x)


class DiffUNet(nn.Module):
    '''Conditional U-Net. Input channel = 6 because x_noisy (3ch) and cond (3ch) are concatenated.'''
    def __init__(self, ic=6, oc=3, ch=64, td=128):
        super().__init__()
        self.te = nn.Sequential(SinusoidalPE(td), nn.Linear(td, td * 4), nn.SiLU(), nn.Linear(td * 4, td))
        self.e1 = DiffResBlock(ic, ch, td)
        self.d1 = nn.Conv2d(ch, ch, 3, 2, 1)
        self.e2 = DiffResBlock(ch, ch * 2, td)
        self.d2 = nn.Conv2d(ch * 2, ch * 2, 3, 2, 1)
        self.mid = DiffResBlock(ch * 2, ch * 2, td)
        self.u2 = nn.ConvTranspose2d(ch * 2, ch * 2, 2, 2)
        self.dec2 = DiffResBlock(ch * 4, ch, td)
        self.u1 = nn.ConvTranspose2d(ch, ch, 2, 2)
        self.dec1 = DiffResBlock(ch * 2, ch, td)
        self.out = nn.Conv2d(ch, oc, 3, padding=1)

    def forward(self, x, cond, t):
        te = self.te(t)
        cond = F.interpolate(cond, size=x.shape[2:], mode='bicubic', align_corners=False)
        inp = torch.cat([x, cond], dim=1)
        e1 = self.e1(inp, te)
        e2 = self.e2(self.d1(e1), te)
        m = self.mid(self.d2(e2), te)
        d2 = self.dec2(torch.cat([self.u2(m), e2], dim=1), te)
        d1 = self.dec1(torch.cat([self.u1(d2), e1], dim=1), te)
        return self.out(d1)


class DiffusionSR(nn.Module):
    '''SR3-style DDPM Super-Resolution.'''
    def __init__(self, T=1000, scale=4):
        super().__init__()
        self.T = T
        self.scale = scale
        self.unet = DiffUNet()
        betas = torch.linspace(1e-4, 0.02, T)
        ab = torch.cumprod(1 - betas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('ab', ab)

    def forward(self, lr, hr):
        t = torch.randint(0, self.T, (lr.size(0),), device=lr.device)
        noise = torch.randn_like(hr)
        ab_t = self.ab[t][:, None, None, None]
        xt = ab_t.sqrt() * hr + (1 - ab_t).sqrt() * noise
        pred_noise = self.unet(xt, lr, t)
        return F.mse_loss(pred_noise, noise)

    @torch.no_grad()
    def sample(self, lr, steps=50):
        hr_h = lr.shape[2] * self.scale
        hr_w = lr.shape[3] * self.scale
        x = torch.randn(lr.shape[0], 3, hr_h, hr_w, device=lr.device)

        stride = self.T // steps
        for tv in range(self.T - 1, -1, -stride):
            tt = torch.full((lr.size(0),), tv, device=lr.device, dtype=torch.long)
            ab = self.ab[tv]
            pn = self.unet(x, lr, tt)
            x0 = ((x - (1 - ab).sqrt() * pn) / ab.sqrt()).clamp(-1, 1)
            if tv > 0:
                abp = self.ab[max(tv - stride, 0)]
                x = abp.sqrt() * x0 + (1 - abp).sqrt() * pn
            else:
                x = x0
        return ((x + 1) / 2).clamp(0, 1)

print('DiffusionSR defined (unchanged from original notebook).')


In [ ]:
# ----------------------------------------------------------------------
# Metrics + MIF (blended) loss -- unchanged from the original notebook
# ----------------------------------------------------------------------
def calculate_entropy(image):
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist = hist / np.sum(hist)
    return -np.sum(hist * np.log2(hist + 1e-7))

def edge_similarity(img1, img2):
    edge1 = sobel(img1)
    edge2 = sobel(img2)
    return np.mean(1 - np.abs(edge1 - edge2))

def blended_loss(perceptual_loss_value, mse_value, l1_value, heatmap_loss_value):
    '''MIF Loss -- unchanged from the original DHA-ESRGAN notebook.'''
    return perceptual_loss_value + mse_value + l1_value + heatmap_loss_value

print('MIF (blended) loss and metric functions ready.')


##  Messidor-2 Dataset Preparation

Reuses the same `InMemorySRDataset` strategy as the original notebook (loads all image bytes into RAM
once so Google Drive disconnects mid-training don't crash the DataLoader -- `num_workers=0` is
mandatory for this reason). HR images are resized to a fixed 256x256 and LR images are generated by
bicubic downsampling x4 (64x64), consistent with the original notebook's degradation model.

An 85/15 train/test split is created from the Messidor-2 images you uploaded so we have a held-out
test set for the generalization and ablation metrics.


In [ ]:
# ----------------------------------------------------------------------
# InMemorySRDataset -- identical strategy to the original notebook
# ----------------------------------------------------------------------
class InMemorySRDataset(Dataset):
    def __init__(self, image_paths, scale_factor=4, fixed_size=(256, 256)):
        self.scale_factor = scale_factor
        self.fixed_size = fixed_size
        self.hr_transform = transforms.Compose([
            transforms.Resize(fixed_size, interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])
        self.lr_transform = transforms.Compose([
            transforms.Resize((fixed_size[0] // scale_factor, fixed_size[1] // scale_factor),
                               interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])
        print(f'Pre-loading {len(image_paths)} images into RAM ...')
        self.image_bytes = []
        failed = 0
        for p in image_paths:
            try:
                with open(p, 'rb') as fh:
                    img_bytes = fh.read()
                    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

                    # Apply HR transform and check dimensions
                    hr_img_check = self.hr_transform(img)
                    if hr_img_check.shape[1] == 0 or hr_img_check.shape[2] == 0:
                        print(f"  Skipped {p}: HR image has zero dimensions after transformation.")
                        failed += 1
                        continue

                    # Apply LR transform and check dimensions
                    lr_img_check = self.lr_transform(img)
                    if lr_img_check.shape[1] == 0 or lr_img_check.shape[2] == 0:
                        print(f"  Skipped {p}: LR image has zero dimensions after transformation.")
                        failed += 1
                        continue

                    # Only append image bytes if both HR and LR transformations result in valid dimensions
                    self.image_bytes.append(img_bytes)
            except Exception as e:
                print(f'  Skipped {p}: {e}')
                failed += 1
        print(f'  Loaded {len(self.image_bytes)} images ({failed} skipped).')

    def __len__(self):
        return len(self.image_bytes)

    def __getitem__(self, idx):
        img = Image.open(io.BytesIO(self.image_bytes[idx])).convert('RGB')
        hr_img = self.hr_transform(img)
        lr_img = self.lr_transform(img)
        return lr_img, hr_img


# ----------------------------------------------------------------------
# Build train/test split from the Messidor-2 folder
# ----------------------------------------------------------------------
VALID_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
all_paths = sorted([
    os.path.join(image_folder, f) for f in os.listdir(image_folder)
    if f.lower().endswith(VALID_EXT)
])
print(f'Found {len(all_paths)} Messidor-2 images at: {image_folder}')
assert len(all_paths) > 0, 'No images found -- check the image_folder path.'

rng = np.random.RandomState(42)
perm = rng.permutation(len(all_paths))
split_idx = int(0.85 * len(all_paths))
train_paths = [all_paths[i] for i in perm[:split_idx]]
test_paths  = [all_paths[i] for i in perm[split_idx:]]
print(f'Train: {len(train_paths)}  |  Test: {len(test_paths)}')

train_dataset = InMemorySRDataset(train_paths)
test_dataset  = InMemorySRDataset(test_paths)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=1,          shuffle=False, num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')


In [ ]:
# --- User to define filenames of images with significant lesion areas ---
# Please replace these with actual filenames from your Messidor-2 test set that you know
# have significant lesion areas. Ensure they are present in the 'test_paths' list.
# The filenames below are examples; you should verify their existence in your dataset.
lesion_image_filenames = [
    'IM001845.JPG',
    '20051130_54077_0400_PP.png',
    'IM000553.JPG',
    'IM001034.JPG',
    '20060410_39476_0200_PP.png',
    '20051116_45465_0200_PP.png',
    '20051020_44616_0100_PP.png',
    '20060410_39433_0200_PP.png',
    '20051020_43883_0100_PP.png',
    '20051213_50993_0100_PP.png',
    'IM001846.JPG',
    '20051130_54078_0400_PP.png',
    # Add more filenames here if you want to explicitly select more than 12 images
    # For example, if you have these in your test_paths:
    # '20051216_47000_0200_PP.png',
    # '20060523_49809_0100_PP.png',
]

# Define the minimum number of samples desired for the lesion test loader
MIN_SAMPLES_FOR_LESION_LOADER = 12 # Ensuring more than 5 samples are always used

# Filter test_paths to include only the specified lesion images
selected_lesion_paths = [p for p in test_paths if os.path.basename(p) in lesion_image_filenames]

# Fallback if specific lesion images are not found or list is empty or insufficient
if not selected_lesion_paths or len(selected_lesion_paths) < MIN_SAMPLES_FOR_LESION_LOADER:
    if not selected_lesion_paths:
        print("WARNING: No specified lesion images found in test_paths.")
    else:
        print(f"WARNING: Only {len(selected_lesion_paths)} specified lesion images found, which is less than the requested {MIN_SAMPLES_FOR_LESION_LOADER}.")

    print(f"Using the first {MIN_SAMPLES_FOR_LESION_LOADER} images from the original test_loader as a fallback to meet sample count.")
    # Take the first N_SAMPLES from the regular test_loader
    temp_lesion_samples = []
    it = iter(test_loader)
    for _ in range(MIN_SAMPLES_FOR_LESION_LOADER):
        try:
            temp_lesion_samples.append(next(it))
        except StopIteration:
            break

    # Create a dummy dataset for these selected samples to mimic the InMemorySRDataset structure
    class FallbackDataset(Dataset):
        def __init__(self, samples): self.samples = samples
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx): return self.samples[idx]

    lesion_test_dataset = FallbackDataset(temp_lesion_samples)
    lesion_test_loader = DataLoader(lesion_test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    N_LESION_SAMPLES = len(temp_lesion_samples)
else:
    print(f"Found {len(selected_lesion_paths)} specified lesion images.")
    lesion_test_dataset = InMemorySRDataset(selected_lesion_paths)
    lesion_test_loader = DataLoader(lesion_test_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    N_LESION_SAMPLES = len(selected_lesion_paths)

print(f"Lesion samples loader ready with {N_LESION_SAMPLES} images.")

In [ ]:
def train_ESRGAN(generator, discriminator, perceptual_loss, dataloader, num_epochs=50, lr=1e-4,
                  save_path="/content/drive/MyDrive/models_metrics/_blended_loss_proposed.csv"):
    generator, discriminator, perceptual_loss = generator.to(device), discriminator.to(device), perceptual_loss.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion_gan = nn.BCELoss()

    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "Loss_D", "Loss_G", "PSNR", "SSIM", "MSE", "MAE", "RMSE", "Entropy", "Edge_Sim", "Blended_Loss"])

    start_epoch = 1   # NOTE: your original notebook hard-coded 150 (a resume artifact from a prior
                      # session on the original dataset). Set to 1 here for a fresh run on Messidor-2.
    for epoch in range(start_epoch, num_epochs + 1):
        total_loss_d, total_loss_g = 0, 0
        total_psnr, total_ssim, total_mse = 0, 0, 0
        total_mae, total_rmse, total_entropy, total_edge_sim, total_blended_loss = 0, 0, 0, 0, 0
        num_batches = len(dataloader)

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            sr_images = generator(lr_images)

            optimizer_D.zero_grad()
            real_labels = torch.ones(hr_images.size(0), 1, device=device)
            fake_labels = torch.zeros(hr_images.size(0), 1, device=device)
            loss_d_real = criterion_gan(discriminator(hr_images), real_labels)
            loss_d_fake = criterion_gan(discriminator(sr_images.detach()), fake_labels)
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            optimizer_D.step()

            optimizer_G.zero_grad()
            perceptual_loss_value = perceptual_loss(sr_images, hr_images)
            mse_value = mse_loss(sr_images, hr_images)
            l1_value  = l1_loss(sr_images, hr_images)

            sr_np = sr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)
            hr_np = hr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)

            psnr_value         = psnr_metric(hr_np, sr_np, data_range=1.0)
            ssim_value         = ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
            entropy_value      = calculate_entropy(cv2.cvtColor((sr_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
            heatmap_loss_value = edge_similarity(hr_np, sr_np)
            blended_value      = blended_loss(perceptual_loss_value, mse_value, l1_value, heatmap_loss_value)

            loss_g = blended_value + 1e-3 * criterion_gan(discriminator(sr_images), real_labels)
            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            optimizer_G.step()

            rmse_value = np.sqrt(mse_value.item())

            total_loss_d        += loss_d.item()
            total_loss_g        += loss_g.item()
            total_psnr          += psnr_value
            total_ssim          += ssim_value
            total_mse           += mse_value.item()
            total_mae           += l1_value.item()
            total_rmse          += rmse_value
            total_entropy       += entropy_value
            total_edge_sim      += heatmap_loss_value
            total_blended_loss  += blended_value.item()

        avg_loss_d       = total_loss_d / num_batches
        avg_loss_g       = total_loss_g / num_batches
        avg_psnr         = total_psnr / num_batches
        avg_ssim         = total_ssim / num_batches
        avg_mse          = total_mse / num_batches
        avg_mae          = total_mae / num_batches
        avg_rmse         = total_rmse / num_batches
        avg_entropy      = total_entropy / num_batches
        avg_edge_sim     = total_edge_sim / num_batches
        avg_blended_loss = total_blended_loss / num_batches

        print(f"Epoch [{epoch}/{num_epochs}] - Loss D: {avg_loss_d:.4f} - Loss G: {avg_loss_g:.4f} - PSNR: {avg_psnr:.4f} - SSIM: {avg_ssim:.4f} - MSE: {avg_mse:.4f} - MAE: {avg_mae:.4f} - RMSE: {avg_rmse:.4f} - Entropy: {avg_entropy:.4f} - Edge Sim: {avg_edge_sim:.4f} - Blended Loss: {avg_blended_loss:.4f}")

        with open(save_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch, avg_loss_d, avg_loss_g, avg_psnr, avg_ssim, avg_mse, avg_mae, avg_rmse, avg_entropy, avg_edge_sim, avg_blended_loss])

    print(f"Training completed. Metrics saved to {save_path}.")
    return generator

print('train_ESRGAN (exact original MIF-loss training loop) ready.')


In [ ]:
def train_ESRGAN_ablation(generator, discriminator, perceptual_loss, dataloader, num_epochs=40, lr=1e-4,
                          save_path="/content/drive/MyDrive/models_metrics/_ablation_metrics.csv", use_mif_components=True):
    generator, discriminator, perceptual_loss = generator.to(device), discriminator.to(device), perceptual_loss.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion_gan = nn.BCELoss()

    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "Loss_D", "Loss_G", "PSNR", "SSIM", "MSE", "MAE", "RMSE", "Entropy", "Edge_Sim", "Blended_Loss"])

    start_epoch = 1
    for epoch in range(start_epoch, num_epochs + 1):
        total_loss_d, total_loss_g = 0, 0
        total_psnr, total_ssim, total_mse = 0, 0, 0
        total_mae, total_rmse, total_entropy, total_edge_sim, total_blended_loss = 0, 0, 0, 0, 0
        num_batches = len(dataloader)

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            sr_images = generator(lr_images)

            optimizer_D.zero_grad()
            real_labels = torch.ones(hr_images.size(0), 1, device=device)
            fake_labels = torch.zeros(hr_images.size(0), 1, device=device)
            loss_d_real = criterion_gan(discriminator(hr_images), real_labels)
            loss_d_fake = criterion_gan(discriminator(sr_images.detach()), fake_labels)
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            optimizer_D.step()

            optimizer_G.zero_grad()

            perceptual_loss_value = perceptual_loss(sr_images, hr_images)
            mse_value = mse_loss(sr_images, hr_images)
            l1_value  = l1_loss(sr_images, hr_images)

            sr_np = sr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)
            hr_np = hr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)

            psnr_value         = psnr_metric(hr_np, sr_np, data_range=1.0)
            ssim_value         = ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
            entropy_value      = calculate_entropy(cv2.cvtColor((sr_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
            heatmap_loss_value = edge_similarity(hr_np, sr_np)

            # For logging, always use the full blended loss definition for consistency
            blended_value_for_log = blended_loss(perceptual_loss_value, mse_value, l1_value, heatmap_loss_value)

            if use_mif_components:
                # Original blended loss for generator training
                loss_g_pixel_perceptual_part = blended_loss_for_log
            else:
                # Only MSE component for generator training when MIF components are excluded
                loss_g_pixel_perceptual_part = mse_value

            loss_g = loss_g_pixel_perceptual_part + 1e-3 * criterion_gan(discriminator(sr_images), real_labels)
            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            optimizer_G.step()

            rmse_value = np.sqrt(mse_value.item())

            total_loss_d        += loss_d.item()
            total_loss_g        += loss_g.item()
            total_psnr          += psnr_value
            total_ssim          += ssim_value
            total_mse           += mse_value.item()
            total_mae           += l1_value.item()
            total_rmse          += rmse_value
            total_entropy       += entropy_value
            total_edge_sim      += heatmap_loss_value
            total_blended_loss  += blended_value_for_log.item() # Use the full blended_value for logging

        avg_loss_d       = total_loss_d / num_batches
        avg_loss_g       = total_loss_g / num_batches
        avg_psnr         = total_psnr / num_batches
        avg_ssim         = total_ssim / num_batches
        avg_mse          = total_mse / num_batches
        avg_mae          = total_mae / num_batches
        avg_rmse         = total_rmse / num_batches
        avg_entropy      = total_entropy / num_batches
        avg_edge_sim     = total_edge_sim / num_batches
        avg_blended_loss = total_blended_loss / num_batches

        print(f"Epoch [{epoch}/{num_epochs}] - Loss D: {avg_loss_d:.4f} - Loss G: {avg_loss_g:.4f} - PSNR: {avg_psnr:.4f} - SSIM: {avg_ssim:.4f} - MSE: {avg_mse:.4f} - MAE: {avg_mae:.4f} - RMSE: {avg_rmse:.4f} - Entropy: {avg_entropy:.4f} - Edge Sim: {avg_edge_sim:.4f} - Blended Loss: {avg_blended_loss:.4f}")

        with open(save_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch, avg_loss_d, avg_loss_g, avg_psnr, avg_ssim, avg_mse, avg_mae, avg_rmse, avg_entropy, avg_edge_sim, avg_blended_loss])

    print(f"Training completed. Metrics saved to {save_path}.")
    return generator

print('Ablation-capable train_ESRGAN_ablation function ready.')

In [ ]:
# ---- Train the Proposed model (Spectral Conv always ON) on Messidor-2 ----
generator       = ProposedGenerator(use_spectral=True)   # spectral conv permanently enabled
discriminator   = Discriminator()
perceptual_loss = PerceptualLoss()

PROPOSED_EPOCHS =40  # match this to your original notebook's epoch budget

proposed_csv = os.path.join(metrics_dir, 'proposed_messidor2_metrics.csv')
generator = train_ESRGAN(generator, discriminator, perceptual_loss, train_loader,
                          num_epochs=PROPOSED_EPOCHS, lr=1e-4, save_path=proposed_csv)

torch.save(generator.state_dict(), os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth'))
print("Proposed model (with Spectral Conv) checkpoint saved.")


In [ ]:
PROPOSED_EPOCHS = 25 # match this to your original notebook's epoch budget


# ---- Train Proposed - no Spectral Conv (MIF ON) ----
print("\n--- Training Proposed model without Spectral Convolution ---")
generator_no_spectral = ProposedGenerator(use_spectral=False).to(device)
discriminator_no_spectral = Discriminator().to(device)
perceptual_loss_no_spectral = PerceptualLoss().to(device)

no_spectral_csv = os.path.join(metrics_dir, 'proposed_no_spectral_messidor2_metrics.csv')
generator_no_spectral = train_ESRGAN_ablation(
    generator_no_spectral, discriminator_no_spectral, perceptual_loss_no_spectral,
    train_loader, num_epochs=PROPOSED_EPOCHS, lr=1e-4, save_path=no_spectral_csv,
    use_mif_components=True
)
torch.save(generator_no_spectral.state_dict(), os.path.join(ckpt_dir, '_ESRGAN_no_spectral.pth'))
print("Proposed model (no Spectral Conv) checkpoint saved.")

In [ ]:
def train_ESRGAN_ablation(generator, discriminator, perceptual_loss, dataloader, num_epochs=40, lr=1e-4,
                          save_path="/content/drive/MyDrive/models_metrics/_ablation_metrics.csv", use_mif_components=True):
    generator, discriminator, perceptual_loss = generator.to(device), discriminator.to(device), perceptual_loss.to(device)

    optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion_gan = nn.BCELoss()

    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "Loss_D", "Loss_G", "PSNR", "SSIM", "MSE", "MAE", "RMSE", "Entropy", "Edge_Sim", "Blended_Loss"])

    start_epoch = 1
    for epoch in range(start_epoch, num_epochs + 1):
        total_loss_d, total_loss_g = 0, 0
        total_psnr, total_ssim, total_mse = 0, 0, 0
        total_mae, total_rmse, total_entropy, total_edge_sim, total_blended_loss = 0, 0, 0, 0, 0
        num_batches = len(dataloader)

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            sr_images = generator(lr_images)

            optimizer_D.zero_grad()
            real_labels = torch.ones(hr_images.size(0), 1, device=device)
            fake_labels = torch.zeros(hr_images.size(0), 1, device=device)
            loss_d_real = criterion_gan(discriminator(hr_images), real_labels)
            loss_d_fake = criterion_gan(discriminator(sr_images.detach()), fake_labels)
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            optimizer_D.step()

            optimizer_G.zero_grad()

            perceptual_loss_value = perceptual_loss(sr_images, hr_images)
            mse_value = mse_loss(sr_images, hr_images)
            l1_value  = l1_loss(sr_images, hr_images)

            sr_np = sr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)
            hr_np = hr_images[0].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)

            psnr_value         = psnr_metric(hr_np, sr_np, data_range=1.0)
            ssim_value         = ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
            entropy_value      = calculate_entropy(cv2.cvtColor((sr_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
            heatmap_loss_value = edge_similarity(hr_np, sr_np)

            # For logging, always use the full blended loss definition for consistency
            blended_value_for_log = blended_loss(perceptual_loss_value, mse_value, l1_value, heatmap_loss_value)

            if use_mif_components:
                # Original blended loss for generator training
                loss_g_pixel_perceptual_part = blended_value_for_log # Corrected typo here
            else:
                # Only MSE component for generator training when MIF components are excluded
                loss_g_pixel_perceptual_part = mse_value

            loss_g = loss_g_pixel_perceptual_part + 1e-3 * criterion_gan(discriminator(sr_images), real_labels)
            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
            optimizer_G.step()

            rmse_value = np.sqrt(mse_value.item())

            total_loss_d        += loss_d.item()
            total_loss_g        += loss_g.item()
            total_psnr          += psnr_value
            total_ssim          += ssim_value
            total_mse           += mse_value.item()
            total_mae           += l1_value.item()
            total_rmse          += rmse_value
            total_entropy       += entropy_value
            total_edge_sim      += heatmap_loss_value
            total_blended_loss  += blended_value_for_log.item() # Use the full blended_value for logging

        avg_loss_d       = total_loss_d / num_batches
        avg_loss_g       = total_loss_g / num_batches
        avg_psnr         = total_psnr / num_batches
        avg_ssim         = total_ssim / num_batches
        avg_mse          = total_mse / num_batches
        avg_mae          = total_mae / num_batches
        avg_rmse         = total_rmse / num_batches
        avg_entropy      = total_entropy / num_batches
        avg_edge_sim     = total_edge_sim / num_batches
        avg_blended_loss = total_blended_loss / num_batches

        print(f"Epoch [{epoch}/{num_epochs}] - Loss D: {avg_loss_d:.4f} - Loss G: {avg_loss_g:.4f} - PSNR: {avg_psnr:.4f} - SSIM: {avg_ssim:.4f} - MSE: {avg_mse:.4f} - MAE: {avg_mae:.4f} - RMSE: {avg_rmse:.4f} - Entropy: {avg_entropy:.4f} - Edge Sim: {avg_edge_sim:.4f} - Blended Loss: {avg_blended_loss:.4f}")

        with open(save_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch, avg_loss_d, avg_loss_g, avg_psnr, avg_ssim, avg_mse, avg_mae, avg_rmse, avg_entropy, avg_edge_sim, avg_blended_loss])

    print(f"Training completed. Metrics saved to {save_path}.")
    return generator

PROPOSED_EPOCHS = 25 # match this to your original notebook's epoch budget


# ---- Train Proposed - no Spectral Conv (MIF ON) ----
print("\n--- Training Proposed model without Spectral Convolution ---")
generator_no_spectral = ProposedGenerator(use_spectral=False).to(device)
discriminator_no_spectral = Discriminator().to(device)
perceptual_loss_no_spectral = PerceptualLoss().to(device)

no_spectral_csv = os.path.join(metrics_dir, 'proposed_no_spectral_messidor2_metrics.csv')
generator_no_spectral = train_ESRGAN_ablation(
    generator_no_spectral, discriminator_no_spectral, perceptual_loss_no_spectral,
    train_loader, num_epochs=PROPOSED_EPOCHS, lr=1e-4, save_path=no_spectral_csv,
    use_mif_components=True
)
torch.save(generator_no_spectral.state_dict(), os.path.join(ckpt_dir, '_ESRGAN_no_spectral.pth'))
print("Proposed model (no Spectral Conv) checkpoint saved.")

In [ ]:
# ---- Train Proposed - no MIF Loss (Spectral ON) ----
print("\n--- Training Proposed model without MIF Loss components ---")
generator_no_mif = ProposedGenerator(use_spectral=True).to(device)
discriminator_no_mif = Discriminator().to(device)
perceptual_loss_no_mif = PerceptualLoss().to(device)

no_mif_csv = os.path.join(metrics_dir, 'proposed_no_mif_messidor2_metrics.csv')
generator_no_mif = train_ESRGAN_ablation(
    generator_no_mif, discriminator_no_mif, perceptual_loss_no_mif,
    train_loader, num_epochs=25, lr=1e-4, save_path=no_mif_csv,
    use_mif_components=False # Key change: exclude MIF components from generator loss
)
torch.save(generator_no_mif.state_dict(), os.path.join(ckpt_dir, '_ESRGAN_no_mif.pth'))
print("Proposed model (no MIF Loss components) checkpoint saved.")

In [ ]:
# ---- Train Proposed - no Spectral Conv, no MIF Loss components ----
print("\n--- Training Proposed model without Spectral Conv and no MIF Loss components ---")
generator_no_spectral_no_mif = ProposedGenerator(use_spectral=False).to(device)
discriminator_no_spectral_no_mif = Discriminator().to(device)
perceptual_loss_no_spectral_no_mif = PerceptualLoss().to(device)

no_spectral_no_mif_csv = os.path.join(metrics_dir, 'proposed_no_spectral_no_mif_messidor2_metrics.csv')
generator_no_spectral_no_mif = train_ESRGAN_ablation(
    generator_no_spectral_no_mif, discriminator_no_spectral_no_mif, perceptual_loss_no_spectral_no_mif,
    train_loader, num_epochs=25, lr=1e-4, save_path=no_spectral_no_mif_csv,
    use_mif_components=False # Key change: exclude MIF components from generator loss
)
torch.save(generator_no_spectral_no_mif.state_dict(), os.path.join(ckpt_dir, '_ESRGAN_no_spectral_no_mif.pth'))
print("Proposed model (no Spectral Conv, no MIF Loss) checkpoint saved.")

In [ ]:
COMPARISON_EPOCHS = 40 # same training budget as the Proposed model, for fairness

def calculate_entropy_orig(image):
    hist = cv2.calcHist([image], [0], None, [256], [0, 256])
    hist = hist / np.sum(hist)
    return float(-np.sum(hist * np.log2(hist + 1e-7)))

def edge_similarity_orig(img1, img2):
    return float(np.mean(1 - np.abs(sobel(img1) - sobel(img2))))

def blended_loss_orig(psnr_value, ssim_value, mae_value):
    '''Metric-only blended score used by the original comparison-model training functions
    (distinct from the generator's MIF training loss in Section 3 -- kept unchanged/as-is).'''
    return (1 - ssim_value) + (1 / (1 + psnr_value)) + mae_value


def _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, model_tag):
    '''Shared body used by all six train_* wrappers below -- logic copied verbatim from the
    original notebook's per-model training functions (only start_epoch=1 replaces start_epoch=150).'''
    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "PSNR", "SSIM", "MSE", "MAE", "RMSE", "Entropy", "Edge_Sim", "Blended_Loss"])

    start_epoch = 1   # FIX: was hard-coded to 150 (resume artifact) in the original notebook
    for epoch in range(start_epoch, num_epochs + 1):
        total_psnr, total_ssim, total_mse, total_mae, total_rmse = 0, 0, 0, 0, 0
        total_entropy, total_edge_sim, total_blended_loss = 0, 0, 0
        num_batches = len(dataloader)

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            optimizer.zero_grad()
            sr_images = model(lr_images)

            hr_images_resized = F.interpolate(hr_images, size=sr_images.shape[2:], mode="bicubic", align_corners=False)
            loss = criterion(sr_images, hr_images_resized)
            loss.backward()
            optimizer.step()

            batch_psnr, batch_ssim, batch_mse, batch_mae, batch_rmse = 0, 0, 0, 0, 0
            batch_entropy, batch_edge_sim, batch_blended_loss = 0, 0, 0

            for i in range(sr_images.shape[0]):
                sr_np = sr_images[i].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)
                hr_np = hr_images_resized[i].permute(1, 2, 0).cpu().detach().numpy().clip(0, 1)

                mae_value = F.l1_loss(sr_images[i], hr_images_resized[i]).item()
                mse_value = F.mse_loss(sr_images[i], hr_images_resized[i]).item()
                rmse_value = np.sqrt(mse_value)
                psnr_value = psnr_metric(hr_np, sr_np, data_range=1.0)
                ssim_value = ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
                entropy_value = calculate_entropy_orig(cv2.cvtColor((sr_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
                edge_sim_value = edge_similarity_orig(hr_np, sr_np)
                blended_value = blended_loss_orig(psnr_value, ssim_value, mae_value)

                batch_psnr += psnr_value
                batch_ssim += ssim_value
                batch_mse += mse_value
                batch_mae += mae_value
                batch_rmse += rmse_value
                batch_entropy += entropy_value
                batch_edge_sim += edge_sim_value
                batch_blended_loss += blended_value

            n = sr_images.shape[0]
            total_psnr += batch_psnr / n
            total_ssim += batch_ssim / n
            total_mse += batch_mse / n
            total_mae += batch_mae / n
            total_rmse += batch_rmse / n
            total_entropy += batch_entropy / n
            total_edge_sim += batch_edge_sim / n
            total_blended_loss += batch_blended_loss / n

        avg_psnr = total_psnr / num_batches
        avg_ssim = total_ssim / num_batches
        avg_mse = total_mse / num_batches
        avg_mae = total_mae / num_batches
        avg_rmse = total_rmse / num_batches
        avg_entropy = total_entropy / num_batches
        avg_edge_sim = total_edge_sim / num_batches
        avg_blended_loss = total_blended_loss / num_batches

        print(f"[{model_tag}] Epoch [{epoch}/{num_epochs}] - PSNR: {avg_psnr:.4f} - SSIM: {avg_ssim:.4f} "
              f"- MSE: {avg_mse:.4f} - MAE: {avg_mae:.4f} - RMSE: {avg_rmse:.4f} "
              f"- Entropy: {avg_entropy:.4f} - Edge Sim: {avg_edge_sim:.4f} - Blended Loss: {avg_blended_loss:.4f}")

        with open(save_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch, avg_psnr, avg_ssim, avg_mse, avg_mae, avg_rmse,
                             avg_entropy, avg_edge_sim, avg_blended_loss])

    print(f"[{model_tag}] Training completed. Metrics saved to {save_path}.")
    return model


def train_vdsr(model, dataloader, num_epochs=165, lr=0.1, save_path=None):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    return _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, 'VDSR')


def train_edsr(model, dataloader, num_epochs=165, lr=1e-4, save_path=None):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    return _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, 'EDSR')


def train_rcan(model, dataloader, num_epochs=165, lr=1e-5, save_path=None):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    return _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, 'RCAN')


def train_swin2sr(model, dataloader, num_epochs=160, lr=1e-4, save_path=None):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    return _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, 'Swin2SR')


def train_mamba(model, dataloader, num_epochs=160, lr=1e-4, save_path=None):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    return _run_simple_training_loop(model, dataloader, optimizer, criterion, num_epochs, save_path, 'VisionMambaSR')


def train_diffusion(model, dataloader, num_epochs=160, lr=2e-4, save_path="/tmp/diffusion_metrics.csv"):
    '''Diffusion training loop -- unchanged in structure from the original notebook
    (epsilon-prediction training loss + 10-step DDIM sample per batch for metric logging).'''
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    with open(save_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Epoch", "PSNR", "SSIM", "MSE", "MAE", "RMSE", "Entropy", "Edge_Sim", "Blended_Loss"])

    start_epoch = 1   # FIX: was hard-coded to 150 in the original notebook
    for epoch in range(start_epoch, num_epochs + 1):
        total_psnr, total_ssim, total_mse, total_mae, total_rmse = 0, 0, 0, 0, 0
        total_entropy, total_edge_sim, total_blended_loss = 0, 0, 0
        num_batches = len(dataloader)

        for lr_images, hr_images in dataloader:
            lr_images, hr_images = lr_images.to(device), hr_images.to(device)

            optimizer.zero_grad()
            diff_loss = model(lr_images, hr_images)
            diff_loss.backward()
            optimizer.step()

            with torch.no_grad():
                sr_images = model.sample(lr_images, steps=10).clamp(0, 1)

            hr_images_resized = F.interpolate(hr_images, size=sr_images.shape[2:], mode="bicubic", align_corners=False)

            batch_psnr, batch_ssim, batch_mse, batch_mae, batch_rmse = 0, 0, 0, 0, 0
            batch_entropy, batch_edge_sim, batch_blended_loss = 0, 0, 0

            for i in range(sr_images.shape[0]):
                sr_np = sr_images[i].permute(1, 2, 0).cpu().numpy().clip(0, 1)
                hr_np = hr_images_resized[i].permute(1, 2, 0).cpu().numpy().clip(0, 1)
                mae_value = F.l1_loss(sr_images[i], hr_images_resized[i]).item()
                mse_value = F.mse_loss(sr_images[i], hr_images_resized[i]).item()
                rmse_value = np.sqrt(mse_value)
                psnr_value = psnr_metric(hr_np, sr_np, data_range=1.0)
                ssim_value = ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
                entropy_value = calculate_entropy_orig(cv2.cvtColor((sr_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
                edge_sim_value = edge_similarity_orig(hr_np, sr_np)
                blended_value = blended_loss_orig(psnr_value, ssim_value, mae_value)

                batch_psnr += psnr_value
                batch_ssim += ssim_value
                batch_mse += mse_value
                batch_mae += mae_value
                batch_rmse += rmse_value
                batch_entropy += entropy_value
                batch_edge_sim += edge_sim_value
                batch_blended_loss += blended_value

            n = sr_images.shape[0]
            total_psnr += batch_psnr / n
            total_ssim += batch_ssim / n
            total_mse += batch_mse / n
            total_mae += batch_mae / n
            total_rmse += batch_rmse / n
            total_entropy += batch_entropy / n
            total_edge_sim += batch_edge_sim / n
            total_blended_loss += batch_blended_loss / n

        avg_psnr = total_psnr / num_batches
        avg_ssim = total_ssim / num_batches
        avg_mse = total_mse / num_batches
        avg_mae = total_mae / num_batches
        avg_rmse = total_rmse / num_batches
        avg_entropy = total_entropy / num_batches
        avg_edge_sim = total_edge_sim / num_batches
        avg_blended_loss = total_blended_loss / num_batches

        print(f"[DiffusionSR] Epoch [{epoch}/{num_epochs}] - PSNR: {avg_psnr:.4f} - SSIM: {avg_ssim:.4f} "
              f"- MSE: {avg_mse:.4f} - MAE: {avg_mae:.4f} - RMSE: {avg_rmse:.4f} "
              f"- Entropy: {avg_entropy:.4f} - Edge Sim: {avg_edge_sim:.4f} - Blended Loss: {avg_blended_loss:.4f}")

        with open(save_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch, avg_psnr, avg_ssim, avg_mse, avg_mae, avg_rmse,
                             avg_entropy, avg_edge_sim, avg_blended_loss])

    print(f"[DiffusionSR] Training completed. Metrics saved to {save_path}.")
    return model

print('Original per-model training functions ready (train_vdsr, train_edsr, train_rcan,')
print('train_swin2sr, train_mamba, train_diffusion).')


In [ ]:
# ---- Train VDSR ----
vdsr_model  = VDSR()
vdsr_csv    = os.path.join(metrics_dir, 'vdsr_metrics.csv')
vdsr_model  = train_vdsr(vdsr_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=vdsr_csv)
torch.save(vdsr_model.state_dict(), os.path.join(ckpt_dir, 'VDSR_model.pth'))


In [ ]:
# ---- Train EDSR ----
edsr_model  = EDSR()
edsr_csv    = os.path.join(metrics_dir, 'edsr_metrics.csv')
edsr_model  = train_edsr(edsr_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=edsr_csv)
torch.save(edsr_model.state_dict(), os.path.join(ckpt_dir, 'EDSR_model.pth'))


In [ ]:
# ---- Train RCAN (reduced groups/blocks, matching the original notebook's Colab-sized config) ----
rcan_model  = RCAN(num_res_groups=3, num_res_blocks=5, feature_size=16)
rcan_csv    = os.path.join(metrics_dir, 'rcan_metrics.csv')
rcan_model  = train_rcan(rcan_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=rcan_csv)
torch.save(rcan_model.state_dict(), os.path.join(ckpt_dir, 'RCAN_model.pth'))


In [ ]:
# ---- Train Swin2SR ----
swin_model  = Swin2SR(in_channels=3, out_channels=3, embed_dim=64, num_heads=4,
                       num_rstb=4, num_swin_blocks=4, window_size=8, scale=4)
swin_csv    = os.path.join(metrics_dir, 'swin2sr_metrics.csv')
swin_model  = train_swin2sr(swin_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=swin_csv)
torch.save(swin_model.state_dict(), os.path.join(ckpt_dir, 'swin2sr_model.pth'))


In [ ]:
# ---- Train VisionMambaSR ----
mamba_model = VisionMambaSR()
mamba_csv   = os.path.join(metrics_dir, 'mamba_metrics.csv')
mamba_model = train_mamba(mamba_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=mamba_csv)
torch.save(mamba_model.state_dict(), os.path.join(ckpt_dir, 'VisionMambaSR_model.pth'))


In [ ]:
# ---- Train DiffusionSR ----
diffusion_model = DiffusionSR()
diffusion_csv   = os.path.join(metrics_dir, 'diffusion_metrics.csv')
diffusion_model = train_diffusion(diffusion_model, train_loader, num_epochs=COMPARISON_EPOCHS, save_path=diffusion_csv)
torch.save(diffusion_model.state_dict(), os.path.join(ckpt_dir, 'DiffusionSR_model.pth'))


In [ ]:
print('Instantiating all comparison models...')

vdsr_model = VDSR().to(device)
edsr_model = EDSR().to(device)
rcan_model = RCAN(num_res_groups=3, num_res_blocks=5, feature_size=16).to(device)
swin_model = Swin2SR(in_channels=3, out_channels=3, embed_dim=64, num_heads=4, num_rstb=4, num_swin_blocks=4, window_size=8, scale=4).to(device)
mamba_model = VisionMambaSR().to(device)
diffusion_model = DiffusionSR().to(device)

print('Comparison models instantiated.')

## Comparison Table & Curves — All 7 Models on Messidor-2

Reproduces the original notebook's "Table II" comparison (PSNR / SSIM / MSE / MAE curves across all
models, plus a final-epoch summary table), now on Messidor-2, with the Proposed model represented by
the Proposed model (Spectral Conv Block always enabled, MIF/blended loss, exactly as trained in Section 5).


In [ ]:
import seaborn as sns

CSV_MAP = {
    'VDSR'         : 'vdsr_metrics.csv',
    'EDSR'         : 'edsr_metrics.csv',
    'RCAN'         : 'rcan_metrics.csv',
    'Swin2SR'      : 'swin2sr_metrics.csv',
    'VisionMambaSR': 'mamba_metrics.csv',
    'DiffusionSR'  : 'diffusion_metrics.csv',
    'Proposed'     : 'proposed_messidor2_metrics.csv',
}
COLORS = {
    'VDSR': '#e41a1c', 'EDSR': '#377eb8', 'RCAN': '#4daf4a', 'Swin2SR': '#984ea3',
    'VisionMambaSR': '#a65628', 'DiffusionSR': '#f781bf', 'Proposed': '#ff7f00',
}
MARKERS = {
    'VDSR': 'o', 'EDSR': 's', 'RCAN': 'D', 'Swin2SR': '^',
    'VisionMambaSR': 'p', 'DiffusionSR': 'h', 'Proposed': '*'
}
LINES = {
    'VDSR': '-', 'EDSR': '-', 'RCAN': '-', 'Swin2SR': '-',
    'VisionMambaSR': '-.', 'DiffusionSR': ':', 'Proposed': '--',
}

master_data = {}
for model, fname in CSV_MAP.items():
    fpath = os.path.join(metrics_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df.columns = [c.strip() for c in df.columns]
        master_data[model] = df
        print(f"  Loaded: {fname}  ({len(df)} epochs)")
    else:
        print(f"  WARNING: {fpath} not found -- {model} skipped")

def plot_master_metric(metric, ylabel, fname, loc='best'):
    fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
    for model, df in master_data.items():
        if metric in df.columns:
            # Removed 'marker=MARKERS[model]' as requested
            sns.lineplot(data=df, x='Epoch', y=metric, color=COLORS[model],
                    linestyle=LINES[model], linewidth=2,
                    markevery=max(1, len(df) // 10), label=model)
    ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc=loc, framealpha=0.9)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    save_path = os.path.join(plots_dir, fname)
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {save_path}")

plot_master_metric('PSNR', 'PSNR (dB)',    'Messidor2_PSNR_Curve.png',    loc='lower right')
plot_master_metric('SSIM', 'SSIM',         'Messidor2_SSIM_Curve.png',    loc='lower right')
plot_master_metric('MSE',  'MSE Loss',     'Messidor2_MSE_Curve.png',     loc='upper right')
plot_master_metric('MAE',  'MAE Loss',     'Messidor2_MAE_Curve.png',     loc='upper right')

rows = []
metric_cols = ['PSNR', 'SSIM', 'MSE', 'MAE'] # Define columns that should not be NaN for the final table

for model in CSV_MAP.keys():
    if model in master_data:
        df_model = master_data[model]
        # Filter out rows where any of the key metrics are NaN, then take the last one
        # If all rows are NaN for a specific metric, this will take the last available row
        # for other metrics, and then individual checks handle the N/A for display.
        last_valid_row = df_model.dropna(subset=metric_cols).iloc[-1] if not df_model.dropna(subset=metric_cols).empty else pd.Series(index=metric_cols, dtype=float)

        rows.append({
            'Model': model,
            'Scale': 4,
            'PSNR': round(float(last_valid_row['PSNR']), 3) if pd.notna(last_valid_row['PSNR']) else 'N/A',
            'SSIM': round(float(last_valid_row['SSIM']), 4) if pd.notna(last_valid_row['SSIM']) else 'N/A',
            'MSE': round(float(last_valid_row['MSE']), 5) if pd.notna(last_valid_row['MSE']) else 'N/A',
            'MAE': round(float(last_valid_row['MAE']), 5) if pd.notna(last_valid_row['MAE']) else 'N/A',
        })
    else:
        rows.append({'Model': model, 'Scale': 4, 'PSNR': 'N/A', 'SSIM': 'N/A', 'MSE': 'N/A', 'MAE': 'N/A'})

table_ii = pd.DataFrame(rows)
print("\n" + "="*68)
print("TABLE II -- Quantitative Comparison (Messidor-2, Scale x4)")
print("="*68)
print(table_ii.to_string(index=False))
print("="*68)

table_ii_path = os.path.join(metrics_dir, 'TABLE_II_messidor2_7models.csv')
table_ii.to_csv(table_ii_path, index=False)
print(f"\nTable saved to: {table_ii_path}")

## Qualitative Comparison Grid — All 7 Models

Visual LR / Ground-Truth-HR / VDSR / EDSR / RCAN / Swin2SR / VisionMambaSR / DiffusionSR / Proposed
panel for a handful of Messidor-2 test images.


In [ ]:
def tensor_to_np(t):
    return t[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)

# Use N_LESION_SAMPLES and lesion_test_loader for qualitative results
# N_SAMPLES = 3 # Original value
# sample_batches = []
# it = iter(test_loader)

sample_batches = []
it = iter(lesion_test_loader)
for _ in range(N_LESION_SAMPLES):
    try:
        sample_batches.append(next(it))
    except StopIteration:
        break

col_titles = ['LR Input', 'VDSR', 'EDSR', 'RCAN', 'Swin2SR',
              'VisionMambaSR', 'DiffusionSR', 'Proposed', 'Ground Truth HR']

for m in [vdsr_model, edsr_model, rcan_model, swin_model, mamba_model, diffusion_model]:
    m.eval()

# Check if 'generator' (the Proposed model) is defined. If not, load it from its checkpoint.
# This assumes ProposedGenerator class and necessary imports are available, which they are.
# Also assumes ckpt_dir and device are defined, which they are from previous cells.
if 'generator' not in locals():
    print("WARNING: 'generator' (Proposed model) not found in current session. Loading from checkpoint.")
    # Instantiate the model with the same parameters as during training
    generator = ProposedGenerator(use_spectral=True).to(device)
    # Define the path to the saved checkpoint
    proposed_ckpt_path = os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth')
    if os.path.exists(proposed_ckpt_path):
        generator.load_state_dict(torch.load(proposed_ckpt_path, map_location=device))
        print(f"Loaded Proposed model from {proposed_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {proposed_ckpt_path}. Please ensure cell 5.1 was executed to train and save the Proposed model.")
        # As a fallback, initialize an untrained model, but warn the user.
        print("Initializing an untrained Proposed model as a fallback. Results may be poor.")

proposed_full = generator   # the single Proposed model trained in Section 5
proposed_full.eval()

fig, axes = plt.subplots(len(sample_batches), len(col_titles),
                          figsize=(2.1 * len(col_titles), 2.3 * len(sample_batches)), dpi=140)
if len(sample_batches) == 1:
    axes = axes[None, :]

for ri, (lr_img_batch, hr_img_batch) in enumerate(sample_batches):
    # Remove the extra batch dimension introduced by DataLoader(FallbackDataset(DataLoader_output))
    lr_img = lr_img_batch.squeeze(0).to(device)
    hr_img = hr_img_batch.squeeze(0).to(device)

    lr_up = F.interpolate(lr_img, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)

    with torch.no_grad():
        sr_vdsr  = vdsr_model(lr_img).clamp(0, 1)   # note: stays at LR resolution (see Section 2 note)
        sr_edsr  = edsr_model(lr_img).clamp(0, 1)   # note: stays at LR resolution
        sr_rcan  = rcan_model(lr_img).clamp(0, 1)   # note: stays at LR resolution
        sr_swin  = swin_model(lr_img).clamp(0, 1)
        sr_mamba = mamba_model(lr_img).clamp(0, 1)
        sr_diff  = diffusion_model.sample(lr_img, steps=10)
        sr_prop  = proposed_full(lr_img).clamp(0, 1)

    panels = [tensor_to_np(lr_up), tensor_to_np(sr_vdsr), tensor_to_np(sr_edsr), tensor_to_np(sr_rcan),
          tensor_to_np(sr_swin), tensor_to_np(sr_mamba), tensor_to_np(sr_diff),
          tensor_to_np(sr_prop), tensor_to_np(hr_img)]

    for ci, panel in enumerate(panels):
        ax = axes[ri, ci]
        ax.imshow(panel)
        ax.axis('off')
        if ri == 0:
            ax.set_title(col_titles[ci], fontsize=9, fontweight='bold')

plt.tight_layout()
qual_path = os.path.join(plots_dir, 'Qualitative_Comparison_Messidor2_All7.png')
plt.savefig(qual_path, dpi=200, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {qual_path}")

In [ ]:
# --- User to define filenames of images for custom qualitative comparison ---
# Please list the filenames (e.g., 'IM001845.JPG') that exist within your `image_folder`.
# These will be loaded and processed for the qualitative plot.
user_input_image_filenames = [

    '20051020_44616_0100_PP.png',
    '20051020_45137_0100_PP.png',
    '20051130_54077_0400_PP.png',
    '20051216_47000_0200_PP.png',
    '20060523_49809_0100_PP.png',
    '20060523_50489_0100_PP.png',
    '20060530_54030_0100_PP.png'
]


# '20051020_44616_0100_PP.png',    '20051020_45137_0100_PP.png',    'IM001845.JPG',    '20051130_54077_0400_PP.png',    '20051216_47000_0200_PP.png',     '20051021_59504_0100_PP.png',  '20051116_58864_0400_PP.png','20060523_49809_0100_PP.png','20060523_50489_0100_PP.png','20060530_54030_0100_PP.png'
# Construct full paths and filter for existing files
user_full_image_paths = []
for fname in user_input_image_filenames:
    full_path = os.path.join(image_folder, fname)
    if os.path.exists(full_path):
        user_full_image_paths.append(full_path)
    else:
        print(f"Warning: Image file not found: {fname}. Skipping.")

if not user_full_image_paths:
    raise ValueError("No valid image files provided or found for qualitative comparison.")

print(f"Found {len(user_full_image_paths)} valid images for custom comparison.")

# Define a simple dataset class to load and preprocess user images on the fly
class UserCustomDataset(Dataset):
    def __init__(self, image_paths, scale_factor=4, fixed_size=(256, 256)):
        self.image_paths = image_paths
        self.scale_factor = scale_factor
        self.fixed_size = fixed_size
        self.hr_transform = transforms.Compose([
            transforms.Resize(fixed_size, interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])
        self.lr_transform = transforms.Compose([
            transforms.Resize((fixed_size[0] // scale_factor, fixed_size[1] // scale_factor),
                               interpolation=Image.BICUBIC),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        hr_img = self.hr_transform(img)
        lr_img = self.lr_transform(img)
        return lr_img, hr_img

user_custom_dataset = UserCustomDataset(user_full_image_paths)
user_custom_test_loader = DataLoader(user_custom_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
N_USER_SAMPLES = len(user_custom_dataset)
print(f"Custom qualitative loader ready with {N_USER_SAMPLES} images.")


In [ ]:
def tensor_to_np(t):
    return t[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)

sample_batches = []
it = iter(user_custom_test_loader)
for _ in range(N_USER_SAMPLES):
    try:
        sample_batches.append(next(it))
    except StopIteration:
        break

col_titles_custom = ['LR Input', 'VDSR', 'RCAN', 'Swin2SR',
                     'VisionMambaSR', 'DiffusionSR', 'Proposed', 'Ground Truth HR']

# --- Ensure all comparison models and proposed model are defined and loaded ---

# Check and load VDSR model if not defined
if 'vdsr_model' not in locals():
    print("WARNING: 'vdsr_model' not found in current session. Attempting to load from checkpoint.")
    vdsr_model = VDSR().to(device)
    vdsr_ckpt_path = os.path.join(ckpt_dir, 'VDSR_model.pth')
    if os.path.exists(vdsr_ckpt_path):
        vdsr_model.load_state_dict(torch.load(vdsr_ckpt_path, map_location=device))
        print(f"Loaded VDSR model from {vdsr_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {vdsr_ckpt_path}. Cannot perform qualitative comparison for VDSR model.")
        vdsr_model = None # Mark as not loaded

# Check and load RCAN model if not defined
if 'rcan_model' not in locals():
    print("WARNING: 'rcan_model' not found in current session. Attempting to load from checkpoint.")
    rcan_model = RCAN(num_res_groups=3, num_res_blocks=5, feature_size=16).to(device)
    rcan_ckpt_path = os.path.join(ckpt_dir, 'RCAN_model.pth')
    if os.path.exists(rcan_ckpt_path):
        rcan_model.load_state_dict(torch.load(rcan_ckpt_path, map_location=device))
        print(f"Loaded RCAN model from {rcan_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {rcan_ckpt_path}. Cannot perform qualitative comparison for RCAN model.")
        rcan_model = None # Mark as not loaded

# Check and load Swin2SR model if not defined
if 'swin_model' not in locals():
    print("WARNING: 'swin_model' not found in current session. Attempting to load from checkpoint.")
    swin_model = Swin2SR(in_channels=3, out_channels=3, embed_dim=64, num_heads=4,
                       num_rstb=4, num_swin_blocks=4, window_size=8, scale=4).to(device)
    swin_ckpt_path = os.path.join(ckpt_dir, 'swin2sr_model.pth')
    if os.path.exists(swin_ckpt_path):
        swin_model.load_state_dict(torch.load(swin_ckpt_path, map_location=device))
        print(f"Loaded Swin2SR model from {swin_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {swin_ckpt_path}. Cannot perform qualitative comparison for Swin2SR model.")
        swin_model = None # Mark as not loaded

# Check and load VisionMambaSR model if not defined
if 'mamba_model' not in locals():
    print("WARNING: 'mamba_model' not found in current session. Attempting to load from checkpoint.")
    mamba_model = VisionMambaSR().to(device)
    mamba_ckpt_path = os.path.join(ckpt_dir, 'VisionMambaSR_model.pth')
    if os.path.exists(mamba_ckpt_path):
        mamba_model.load_state_dict(torch.load(mamba_ckpt_path, map_location=device))
        print(f"Loaded VisionMambaSR model from {mamba_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {mamba_ckpt_path}. Cannot perform qualitative comparison for VisionMambaSR model.")
        mamba_model = None # Mark as not loaded

# Check and load DiffusionSR model if not defined
if 'diffusion_model' not in locals():
    print("WARNING: 'diffusion_model' not found in current session. Attempting to load from checkpoint.")
    diffusion_model = DiffusionSR().to(device)
    diffusion_ckpt_path = os.path.join(ckpt_dir, 'DiffusionSR_model.pth')
    if os.path.exists(diffusion_ckpt_path):
        diffusion_model.load_state_dict(torch.load(diffusion_ckpt_path, map_location=device))
        print(f"Loaded DiffusionSR model from {diffusion_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {diffusion_ckpt_path}. Cannot perform qualitative comparison for DiffusionSR model.")
        diffusion_model = None # Mark as not loaded

# Check if 'generator' (the Proposed model) is defined and loaded
if 'generator' not in locals():
    print("WARNING: 'generator' (Proposed model) not found in current session. Attempting to load from checkpoint.")
    generator = ProposedGenerator(use_spectral=True).to(device)
    proposed_ckpt_path = os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth')
    if os.path.exists(proposed_ckpt_path):
        generator.load_state_dict(torch.load(proposed_ckpt_path, map_location=device))
        print(f"Loaded Proposed model from {proposed_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {proposed_ckpt_path}. Cannot perform qualitative comparison for Proposed model.")
        generator = None # Mark as not loaded

proposed_full = generator # Use the already loaded generator
if proposed_full: # Only call eval if the model was successfully loaded
    proposed_full.eval()

# Ensure models are in evaluation mode (excluding EDSR)
for m in [vdsr_model, rcan_model, swin_model, mamba_model, diffusion_model]:
    # Only call eval if the model was successfully loaded (i.e., not None)
    if m: m.eval()

fig, axes = plt.subplots(len(sample_batches), len(col_titles_custom),
                          figsize=(2.1 * len(col_titles_custom), 2.3 * len(sample_batches)), dpi=140)

# Handle case where there's only one row of plots (single image)
if len(sample_batches) == 1:
    axes = axes[None, :]

for ri, (lr_img_batch, hr_img_batch) in enumerate(sample_batches):
    # Fix: Keep batch dimension (batch_size=1 from DataLoader)
    lr_img = lr_img_batch.to(device)
    hr_img = hr_img_batch.to(device)

    # F.interpolate input now correctly has batch dimension from lr_img
    lr_up = F.interpolate(lr_img, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)

    with torch.no_grad():
        # All models now receive lr_img with batch dimension (1, C, H, W)
        sr_vdsr  = vdsr_model(lr_img).clamp(0, 1) if vdsr_model else torch.zeros_like(hr_img)
        # sr_edsr = edsr_model(lr_img).clamp(0, 1) # EDSR is intentionally excluded
        sr_rcan  = rcan_model(lr_img).clamp(0, 1) if rcan_model else torch.zeros_like(hr_img)
        sr_swin  = swin_model(lr_img).clamp(0, 1) if swin_model else torch.zeros_like(hr_img)
        sr_mamba = mamba_model(lr_img).clamp(0, 1) if mamba_model else torch.zeros_like(hr_img)
        sr_diff  = diffusion_model.sample(lr_img, steps=10) if diffusion_model else torch.zeros_like(hr_img)
        # Adjust proposed_full condition to return (1,C,H,W) in fallback
        sr_prop  = proposed_full(lr_img).clamp(0, 1) if proposed_full else torch.zeros_like(hr_img)

    # Construct panels list, explicitly excluding EDSR
    panels = [
        tensor_to_np(lr_up),
        tensor_to_np(sr_vdsr),
        tensor_to_np(sr_rcan),
        tensor_to_np(sr_swin),
        tensor_to_np(sr_mamba),
        tensor_to_np(sr_diff),
        tensor_to_np(sr_prop),
        tensor_to_np(hr_img)
    ]

    for ci, panel in enumerate(panels):
        ax = axes[ri, ci]
        ax.imshow(panel)
        ax.axis('off')
        if ri == 0:
            ax.set_title(col_titles_custom[ci], fontsize=9, fontweight='bold')

plt.tight_layout()
qual_path_custom = os.path.join(plots_dir, 'Qualitative_Comparison_Messidor2_User_Custom.png')
plt.savefig(qual_path_custom, dpi=400, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved custom qualitative comparison: {qual_path_custom}")

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def find_lesion_bbox(hr_np, box_size=64, exclude_optic_disc=True, min_blob_area=5):
    """
    Auto-detect a lesion crop box on the Ground-Truth HR image.

    Finds the most prominent bright blob (hard exudate / cotton-wool-spot
    proxy) that is NOT the optic disc (assumed to be the single largest
    bright blob), and returns a (x, y, w, h) box of size box_size x box_size
    centered on it. Falls back to a center crop if no blob is found.

    A morphological closing step merges fragmented top-hat responses (e.g.
    thin boundary slivers around the optic disc) into single connected
    blobs, and `min_blob_area` filters out tiny noise specks, so the
    "second-largest blob" heuristic doesn't accidentally lock onto an
    optic-disc edge fragment instead of the real lesion.
    """
    img_u8 = (hr_np * 255).astype(np.uint8)
    gray = cv2.cvtColor(img_u8, cv2.COLOR_RGB2GRAY)

    _, mask = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))

    # For darker lesions (hemorrhages), BLACKHAT is more appropriate
    kernel_lesion_detection = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)) # Increased kernel size for better hemorrhage detection
    lesion_highlight = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel_lesion_detection)

    # Apply the existing mask to the highlighted lesions
    lesion_highlight = cv2.bitwise_and(lesion_highlight, lesion_highlight, mask=mask)

    # Threshold the blackhat output to get binary lesions.
    # The threshold '15' is lowered to improve sensitivity to darker lesions.
    _, thresh = cv2.threshold(lesion_highlight, 15, 255, cv2.THRESH_BINARY)

    # Merge fragmented/boundary-sliver blobs into solid connected regions
    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, close_kernel)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh, connectivity=8)
    h, w = gray.shape

    if num_labels <= 1:
        return (w // 2 - box_size // 2, h // 2 - box_size // 2, box_size, box_size)

    areas = stats[1:, cv2.CC_STAT_AREA]
    valid = np.where(areas >= min_blob_area)[0]
    if len(valid) == 0:
        return (w // 2 - box_size // 2, h // 2 - box_size // 2, box_size, box_size)

    order = valid[np.argsort(areas[valid])[::-1]]

    if exclude_optic_disc and len(order) > 1:
        lesion_idx = order[1] + 1
    else:
        lesion_idx = order[0] + 1

    cx, cy = centroids[lesion_idx]
    x = int(np.clip(cx - box_size / 2, 0, w - box_size))
    y = int(np.clip(cy - box_size / 2, 0, h - box_size))
    return (x, y, box_size, box_size)


def crop_and_zoom(img_np, bbox, zoom=4):
    x, y, w, h = bbox
    crop = img_np[y:y + h, x:x + w]
    # Add a check for empty crop before resizing
    if crop.shape[0] == 0 or crop.shape[1] == 0:
        print(f"Warning: Cropped image has zero dimensions (shape: {crop.shape}). Returning black image.")
        # Return a black image of the target zoomed size to avoid crash
        return np.zeros((h * zoom, w * zoom, img_np.shape[2]), dtype=img_np.dtype)
    zoomed = cv2.resize(crop, (w * zoom, h * zoom), interpolation=cv2.INTER_CUBIC)
    return zoomed.clip(0, 1)   # cubic interpolation can slightly overshoot [0,1]


def tensor_to_np(t):
    # Ensure tensor has batch dimension before permute, if it's already removed, handle it
    if t.dim() == 3: # (C, H, W)
        return t.permute(1, 2, 0).cpu().numpy().clip(0, 1)
    elif t.dim() == 4: # (N, C, H, W)
        return t[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)
    else:
        raise ValueError(f"Unexpected tensor dimension: {t.dim()}")


# ── Column order (as requested): LR, VDSR, EDSR, RCAN, Swin2SR,
#    VisionMambaSR, DiffusionSR, Proposed, Ground Truth HR ───────────
col_titles = ['Original Image n(GT + box)', 'LR Input', 'VDSR', 'EDSR', 'RCAN',
              'Swin2SR', 'VisionMambaSR', 'DiffusionSR', 'Proposed', 'Ground Truth HR']

BOX_SIZE = 64   # lesion crop size in pixels (at 256x256 HR resolution)
ZOOM     = 4    # upscale factor for the zoomed crop strip

# --- Ensure all comparison models and proposed model are defined and loaded ---

# Check and load VDSR model if not defined
if 'vdsr_model' not in locals():
    print("WARNING: 'vdsr_model' not found in current session. Attempting to load from checkpoint.")
    vdsr_model = VDSR().to(device)
    vdsr_ckpt_path = os.path.join(ckpt_dir, 'VDSR_model.pth')
    if os.path.exists(vdsr_ckpt_path):
        vdsr_model.load_state_dict(torch.load(vdsr_ckpt_path, map_location=device))
        print(f"Loaded VDSR model from {vdsr_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {vdsr_ckpt_path}. Cannot perform qualitative comparison for VDSR model.")
        vdsr_model = None # Mark as not loaded

# Check and load EDSR model if not defined
if 'edsr_model' not in locals():
    print("WARNING: 'edsr_model' not found in current session. Attempting to load from checkpoint.")
    edsr_model = EDSR().to(device)
    edsr_ckpt_path = os.path.join(ckpt_dir, 'EDSR_model.pth')
    if os.path.exists(edsr_ckpt_path):
        edsr_model.load_state_dict(torch.load(edsr_ckpt_path, map_location=device))
        print(f"Loaded EDSR model from {edsr_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {edsr_ckpt_path}. Cannot perform qualitative comparison for EDSR model.")
        edsr_model = None # Mark as not loaded

# Check and load RCAN model if not defined
if 'rcan_model' not in locals():
    print("WARNING: 'rcan_model' not found in current session. Attempting to load from checkpoint.")
    rcan_model = RCAN(num_res_groups=3, num_res_blocks=5, feature_size=16).to(device)
    rcan_ckpt_path = os.path.join(ckpt_dir, 'RCAN_model.pth')
    if os.path.exists(rcan_ckpt_path):
        rcan_model.load_state_dict(torch.load(rcan_ckpt_path, map_location=device))
        print(f"Loaded RCAN model from {rcan_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {rcan_ckpt_path}. Cannot perform qualitative comparison for RCAN model.")
        rcan_model = None # Mark as not loaded

# Check and load Swin2SR model if not defined
if 'swin_model' not in locals():
    print("WARNING: 'swin_model' not found in current session. Attempting to load from checkpoint.")
    swin_model = Swin2SR(in_channels=3, out_channels=3, embed_dim=64, num_heads=4,
                       num_rstb=4, num_swin_blocks=4, window_size=8, scale=4).to(device)
    swin_ckpt_path = os.out.join(ckpt_dir, 'swin2sr_model.pth')
    if os.path.exists(swin_ckpt_path):
        swin_model.load_state_dict(torch.load(swin_ckpt_path, map_location=device))
        print(f"Loaded Swin2SR model from {swin_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {swin_ckpt_path}. Cannot perform qualitative comparison for Swin2SR model.")
        swin_model = None # Mark as not loaded

# Check and load VisionMambaSR model if not defined
if 'mamba_model' not in locals():
    print("WARNING: 'mamba_model' not found in current session. Attempting to load from checkpoint.")
    mamba_model = VisionMambaSR().to(device)
    mamba_ckpt_path = os.path.join(ckpt_dir, 'VisionMambaSR_model.pth')
    if os.path.exists(mamba_ckpt_path):
        mamba_model.load_state_dict(torch.load(mamba_ckpt_path, map_location=device))
        print(f"Loaded VisionMambaSR model from {mamba_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {mamba_ckpt_path}. Cannot perform qualitative comparison for VisionMambaSR model.")
        mamba_model = None # Mark as not loaded

# Check and load DiffusionSR model if not defined
if 'diffusion_model' not in locals():
    print("WARNING: 'diffusion_model' not found in current session. Attempting to load from checkpoint.")
    diffusion_model = DiffusionSR().to(device)
    diffusion_ckpt_path = os.path.join(ckpt_dir, 'DiffusionSR_model.pth')
    if os.path.exists(diffusion_ckpt_path):
        diffusion_model.load_state_dict(torch.load(diffusion_ckpt_path, map_location=device))
        print(f"Loaded DiffusionSR model from {diffusion_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {diffusion_ckpt_path}. Cannot perform qualitative comparison for DiffusionSR model.")
        diffusion_model = None # Mark as not loaded

# Check if 'generator' (the Proposed model) is defined and loaded
if 'generator' not in locals():
    print("WARNING: 'generator' (Proposed model) not found in current session. Attempting to load from checkpoint.")
    generator = ProposedGenerator(use_spectral=True).to(device)
    proposed_ckpt_path = os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth')
    if os.path.exists(proposed_ckpt_path):
        generator.load_state_dict(torch.load(proposed_ckpt_path, map_location=device))
        print(f"Loaded Proposed model from {proposed_ckpt_path}")
    else:
        print(f"Error: Checkpoint not found at {proposed_ckpt_path}. Cannot perform qualitative comparison for Proposed model.")
        generator = None # Mark as not loaded

# Create a list of models that were successfully loaded to iterate over for eval mode
models_to_eval = []
if vdsr_model: models_to_eval.append(vdsr_model)
if edsr_model: models_to_eval.append(edsr_model)
if rcan_model: models_to_eval.append(rcan_model)
if swin_model: models_to_eval.append(swin_model)
if mamba_model: models_to_eval.append(mamba_model)
if diffusion_model: models_to_eval.append(diffusion_model)

for m in models_to_eval:
    m.eval()
if generator: generator.eval() # Ensure Proposed model is in eval mode

# Initialize sample_batches (missing from the provided code)
sample_batches = []
it = iter(lesion_test_loader)
for _ in range(N_LESION_SAMPLES):
    try:
        sample_batches.append(next(it))
    except StopIteration:
        break

n_rows = len(sample_batches)
fig, axes = plt.subplots(n_rows, len(col_titles),
                          figsize=(2.0 * len(col_titles), 2.2 * n_rows), dpi=150)
if n_rows == 1:
    axes = axes[None, :]

for ri, (lr_img_batch, hr_img_batch) in enumerate(sample_batches):
    # Remove the extra batch dimension added by DataLoader wrapping an already-batched tensor
    lr_img, hr_img = lr_img_batch.squeeze(0).to(device), hr_img_batch.squeeze(0).to(device)
    # lr_up is no longer needed if we are displaying the actual LR image cropped
    # lr_up = F.interpolate(lr_img, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)

    with torch.no_grad():
        # Get LR-resolution outputs for models that don't internally upsample
        sr_vdsr_lr  = vdsr_model(lr_img) if vdsr_model else torch.zeros_like(lr_img)
        sr_edsr_lr  = edsr_model(lr_img) if edsr_model else torch.zeros_like(lr_img)
        sr_rcan_lr  = rcan_model(lr_img) if rcan_model else torch.zeros_like(lr_img)

        # Get HR-resolution outputs for models that internally upsample
        sr_swin  = swin_model(lr_img) if swin_model else torch.zeros_like(hr_img)
        sr_mamba = mamba_model(lr_img) if mamba_model else torch.zeros_like(hr_img)
        # Diffusion model already handles its own clamping in sample method
        sr_diff  = diffusion_model.sample(lr_img, steps=10) if diffusion_model else torch.zeros_like(hr_img)
        sr_prop  = generator(lr_img) if generator else torch.zeros_like(hr_img)

    # Upsample LR-resolution model outputs to HR size before cropping
    sr_vdsr = F.interpolate(sr_vdsr_lr, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)
    sr_edsr = F.interpolate(sr_edsr_lr, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)
    sr_rcan = F.interpolate(sr_rcan_lr, size=hr_img.shape[-2:], mode='bicubic', align_corners=False)

    hr_np    = tensor_to_np(hr_img)
    # For the LR Input column, we need the original 64x64 lr_img
    lr_img_np = tensor_to_np(lr_img)
    vdsr_np  = tensor_to_np(sr_vdsr)
    edsr_np  = tensor_to_np(sr_edsr)
    rcan_np  = tensor_to_np(sr_rcan)
    swin_np  = tensor_to_np(sr_swin)
    mamba_np = tensor_to_np(sr_mamba)
    diff_np  = tensor_to_np(sr_diff)
    prop_np  = tensor_to_np(sr_prop)

    # ── Auto-detect lesion crop box from the Ground-Truth HR image ──
    bbox = find_lesion_bbox(hr_np, box_size=BOX_SIZE)
    x, y, w, h = bbox

    # Calculate scaled bbox for the 64x64 LR image
    lr_x = x // ZOOM
    lr_y = y // ZOOM
    lr_w = w // ZOOM
    lr_h = h // ZOOM
    lr_bbox = (lr_x, lr_y, lr_w, lr_h)

    # ── Panel 0: full GT image with the crop region boxed in yellow ──
    ax0 = axes[ri, 0]
    ax0.imshow(hr_np)
    rect = patches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor='yellow', facecolor='none')
    ax0.add_patch(rect)
    ax0.axis('off')
    if ri == 0:
        ax0.set_title(col_titles[0], fontsize=8, fontweight='bold')

    # ── Panels 1..N: zoomed crops from the SAME box, one per method ──
    # The first element here corresponds to 'LR Input' (col_titles[1])
    # Use the correctly cropped and zoomed original LR image for the 'LR Input' panel
    lr_input_for_plot = crop_and_zoom(lr_img_np, lr_bbox, zoom=ZOOM)

    crop_sources = [lr_input_for_plot, vdsr_np, edsr_np, rcan_np, swin_np, mamba_np, diff_np, prop_np, hr_np]
    for ci, src in enumerate(crop_sources, start=1):
        # For all other sources, the bbox (derived from HR) is directly applicable
        # as they are all 256x256 images.
        zoomed = crop_and_zoom(src, bbox, zoom=ZOOM)
        ax = axes[ri, ci]
        ax.imshow(zoomed)
        ax.axis('off')
        if ri == 0:
            ax.set_title(col_titles[ci], fontsize=8, fontweight='bold')

plt.suptitle(
    "Lesion-Focused Qualitative Comparison on Messidor-2\n"
    "Leftmost column: Ground-Truth HR with the auto-detected lesion region boxed in yellow. "
    "Remaining columns: zoomed crops of that exact region from each method's output.",
    fontsize=9, y=1.01
)
plt.tight_layout()

lesion_fig_path = os.path.join(plots_dir, 'Lesion_Focused_Comparison_Messidor2.png')
plt.savefig(lesion_fig_path, dpi=3200, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved: {lesion_fig_path}")

## Ablation Study Qualitative Comparison

### Ablation Study Qualitative Comparison (Baseline Excluded)

In [ ]:
import torch.nn.functional as F

# Helper function to convert a PyTorch tensor to a NumPy array for plotting
def tensor_to_np(t):
    # If tensor has a batch dimension, take the first item and remove batch dim
    if t.dim() == 4: # (N, C, H, W)
        return t[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)
    elif t.dim() == 3: # (C, H, W)
        return t.permute(1, 2, 0).cpu().numpy().clip(0, 1)
    else:
        raise ValueError(f"Unexpected tensor dimension: {t.dim()}")

# Select 2 sample images from the lesion_test_loader for visualization
sample_ablation_batches = []
it_ablation = iter(lesion_test_loader)
for _ in range(2):
    try:
        sample_ablation_batches.append(next(it_ablation))
    except StopIteration:
        print("Not enough samples in lesion_test_loader for ablation visualization.")
        break

if not sample_ablation_batches:
    raise ValueError("No sample images available for ablation study visualization.")

# Define new, clearer column titles for the desired ablation configurations
col_titles_ablation = [
    'LR',
    'GHR',
    'Spectral Conv Only',
    'MIF Loss Only',
    'Full Proposed (SC + MIF)'
]

# --- Ensure all ablation models are defined and loaded ---
# This section assumes ProposedGenerator and Discriminator classes are already defined.

# Proposed model (full, with Spectral Conv and MIF components) - original `generator`
# We still need to load it if not in memory.
if 'proposed_full_model' not in locals() or proposed_full_model is None:
    if 'generator' not in locals() or generator is None:
        print("WARNING: 'generator' (Proposed model) not found. Loading from checkpoint.")
        generator = ProposedGenerator(use_spectral=True).to(device)
        proposed_ckpt_path = os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth')
        if os.path.exists(proposed_ckpt_path):
            generator.load_state_dict(torch.load(proposed_ckpt_path, map_location=device))
            print(f"Loaded Proposed (Full) model from {proposed_ckpt_path}")
        else:
            print(f"Error: Checkpoint not found at {proposed_ckpt_path}. Cannot load Proposed (Full).")
            generator = None
    proposed_full_model = generator
if proposed_full_model: proposed_full_model.eval()

# Proposed - no Spectral Conv (with MIF components) (referred to as `sr_mif_only` in previous code)
if 'generator_no_spectral' not in locals() or generator_no_spectral is None:
    print("WARNING: 'generator_no_spectral' not found. Loading from checkpoint.")
    generator_no_spectral = ProposedGenerator(use_spectral=False).to(device)
    ckpt_path_mif_only = os.path.join(ckpt_dir, 'ablation_+MIFLoss.pth') # Correct filename
    if os.path.exists(ckpt_path_mif_only):
        generator_no_spectral.load_state_dict(torch.load(ckpt_path_mif_only, map_location=device))
        print(f"Loaded Proposed (No Spectral Conv, MIF Loss ON) model from {ckpt_path_mif_only}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_mif_only}. Cannot visualize Proposed (No Spectral Conv).")
        generator_no_spectral = None
if generator_no_spectral: generator_no_spectral.eval()

# Proposed - no MIF Loss components (with Spectral Conv) (referred to as `sr_spec_only` in previous code)
if 'generator_no_mif' not in locals() or generator_no_mif is None:
    print("WARNING: 'generator_no_mif' not found. Loading from checkpoint.")
    generator_no_mif = ProposedGenerator(use_spectral=True).to(device)
    ckpt_path_spec_only = os.path.join(ckpt_dir, 'ablation_+SpectralConv.pth') # Correct filename
    if os.path.exists(ckpt_path_spec_only):
        generator_no_mif.load_state_dict(torch.load(ckpt_path_spec_only, map_location=device))
        print(f"Loaded Proposed (Spectral Conv ON, No MIF Loss) model from {ckpt_path_spec_only}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_spec_only}. Cannot visualize Proposed (No MIF Loss).")
        generator_no_mif = None
if generator_no_mif: generator_no_mif.eval()

# Proposed - no Spectral Conv, no MIF Loss (Baseline) -- This model will be loaded but not plotted
if 'generator_no_spectral_no_mif' not in locals() or generator_no_spectral_no_mif is None:
    print("WARNING: 'generator_no_spectral_no_mif' (Baseline) not found. Loading from checkpoint.")
    generator_no_spectral_no_mif = ProposedGenerator(use_spectral=False).to(device)
    ckpt_path_baseline = os.path.join(ckpt_dir, 'ablation_Baseline.pth') # Assuming this is the correct filename
    if os.path.exists(ckpt_path_baseline):
        generator_no_spectral_no_mif.load_state_dict(torch.load(ckpt_path_baseline, map_location=device))
        print(f"Loaded Proposed (Baseline) model from {ckpt_path_baseline}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_baseline}. Baseline model will be unavailable.")
        generator_no_spectral_no_mif = None
if generator_no_spectral_no_mif: generator_no_spectral_no_mif.eval()


num_rows = len(sample_ablation_batches)
num_cols = len(col_titles_ablation) # This will now be 5 columns

fig, axes = plt.subplots(num_rows, num_cols,
                          figsize=(2.5 * num_cols, 2.5 * num_rows), dpi=140)

# Handle case where there's only one row of plots (single image)
if num_rows == 1:
    axes = axes[None, :]

for ri, (lr_img_batch, hr_img_batch) in enumerate(sample_ablation_batches):
    # Fix: Squeeze the extra batch dimension. DataLoader(FallbackDataset(DataLoader_output)) adds an extra dim.
    lr_img = lr_img_batch.squeeze(0).to(device)
    hr_img = hr_img_batch.squeeze(0).to(device)

    # Upsample LR input for display purposes
    lr_up = F.interpolate(lr_img, size=hr_img.shape[-2:], mode='bicubic', align_corners=False).clamp(0, 1)

    with torch.no_grad():
        # 1. Baseline: No Spectral Conv, No MIF Loss (loaded but not plotted as per user request)
        # sr_baseline = generator_no_spectral_no_mif(lr_img).clamp(0, 1) if generator_no_spectral_no_mif else lr_up

        # 2. Spectral Conv Only: Spectral Conv ON, No MIF Loss
        sr_spec_only = generator_no_mif(lr_img).clamp(0, 1) if generator_no_mif else lr_up

        # 3. MIF Loss Only: No Spectral Conv, MIF Loss ON
        sr_mif_only = generator_no_spectral(lr_img).clamp(0, 1) if generator_no_spectral else lr_up

        # 4. Full Proposed: Spectral Conv ON, MIF Loss ON
        sr_full_proposed = proposed_full_model(lr_img).clamp(0, 1) if proposed_full_model else lr_up

    panels = [
        tensor_to_np(lr_up),
        tensor_to_np(hr_img),
        tensor_to_np(sr_spec_only),
        tensor_to_np(sr_mif_only),
        tensor_to_np(sr_full_proposed)
    ]

    for ci, panel in enumerate(panels):
        ax = axes[ri, ci]
        ax.imshow(panel)
        ax.axis('off')
        if ri == 0:
            ax.set_title(col_titles_ablation[ci], fontsize=9, fontweight='bold')

plt.tight_layout()
qual_ablation_path_no_baseline = os.path.join(plots_dir, 'Qualitative_Ablation_Comparison_Messidor2_No_Baseline.png')
plt.savefig(qual_ablation_path_no_baseline, dpi=400, bbox_inches='tight')
plt.show()
plt.close()
print(f"Saved ablation qualitative comparison (excluding baseline): {qual_ablation_path_no_baseline}")

## Quantitative Ablation Study (PSNR and SSIM)

Now we perform a quantitative ablation study by evaluating the PSNR and SSIM values of all four Proposed model variants on the test set. This will provide objective metrics for comparing the impact of the Spectral Convolution and MIF Loss components.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

# --- Load all ablation models if not already in memory ---

# Ensure Proposed (Full) model is loaded
if 'proposed_full_model' not in locals() or proposed_full_model is None:
    if 'generator' not in locals() or generator is None:
        print("WARNING: 'generator' (Proposed model) not found. Attempting to load from checkpoint.")
        generator = ProposedGenerator(use_spectral=True).to(device)
        proposed_ckpt_path = os.path.join(ckpt_dir, '_ESRGAN_messidor2.pth')
        if os.path.exists(proposed_ckpt_path):
            generator.load_state_dict(torch.load(proposed_ckpt_path, map_location=device))
            print(f"Loaded Proposed (Full) model from {proposed_ckpt_path}")
        else:
            print(f"Error: Checkpoint not found at {proposed_ckpt_path}. Cannot load Proposed (Full).")
            generator = None
    proposed_full_model = generator
if proposed_full_model: proposed_full_model.eval()

# Ensure 'generator_no_spectral' (MIF Loss present, Spectral Conv absent) is loaded
if 'generator_no_spectral' not in locals() or generator_no_spectral is None:
    print("WARNING: 'generator_no_spectral' not found in current session. Attempting to load from checkpoint.")
    generator_no_spectral = ProposedGenerator(use_spectral=False).to(device)
    ckpt_path_mif_only = os.path.join(ckpt_dir, 'ablation_+MIFLoss.pth') # Correct filename
    if os.path.exists(ckpt_path_mif_only):
        generator_no_spectral.load_state_dict(torch.load(ckpt_path_mif_only, map_location=device))
        print(f"Loaded Proposed - no Spectral Conv from {ckpt_path_mif_only}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_mif_only}. Cannot evaluate Proposed - no Spectral Conv.")
        generator_no_spectral = None
if generator_no_spectral: generator_no_spectral.eval()

# Ensure 'generator_no_mif' (Spectral Conv present, MIF Loss absent) is loaded
if 'generator_no_mif' not in locals() or generator_no_mif is None:
    print("WARNING: 'generator_no_mif' not found in current session. Attempting to load from checkpoint.")
    generator_no_mif = ProposedGenerator(use_spectral=True).to(device)
    ckpt_path_spec_only = os.path.join(ckpt_dir, 'ablation_+SpectralConv.pth') # Correct filename
    if os.path.exists(ckpt_path_spec_only):
        generator_no_mif.load_state_dict(torch.load(ckpt_path_spec_only, map_location=device))
        print(f"Loaded Proposed - no MIF Loss components from {ckpt_path_spec_only}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_spec_only}. Cannot evaluate Proposed - no MIF Loss components.")
        generator_no_mif = None
if generator_no_mif: generator_no_mif.eval()

# Ensure 'generator_no_spectral_no_mif' (Baseline) is loaded
if 'generator_no_spectral_no_mif' not in locals() or generator_no_spectral_no_mif is None:
    print("WARNING: 'generator_no_spectral_no_mif' (Baseline) not found in current session. Attempting to load from checkpoint.")
    generator_no_spectral_no_mif = ProposedGenerator(use_spectral=False).to(device)
    ckpt_path_baseline = os.path.join(ckpt_dir, 'ablation_Baseline.pth') # Correct filename
    if os.path.exists(ckpt_path_baseline):
        generator_no_spectral_no_mif.load_state_dict(torch.load(ckpt_path_baseline, map_location=device))
        print(f"Loaded Proposed (Baseline) model from {ckpt_path_baseline}")
    else:
        print(f"Error: Checkpoint not found at {ckpt_path_baseline}. Cannot evaluate Proposed (Baseline).")
        generator_no_spectral_no_mif = None
if generator_no_spectral_no_mif: generator_no_spectral_no_mif.eval()


models_to_evaluate = {
    'Proposed (Full)': proposed_full_model,
    'Proposed - Spectral Conv Only': generator_no_mif, # Spectral Conv ON, MIF Loss OFF
    'Proposed - MIF Loss Only': generator_no_spectral,  # Spectral Conv OFF, MIF Loss ON
    'Proposed - Baseline (No SC, No MIF)': generator_no_spectral_no_mif # Spectral Conv OFF, MIF Loss OFF
}

results = []

print("\n--- Evaluating Ablation Models ---")
for model_name, model in models_to_evaluate.items():
    if model is None: # Skip if model failed to load
        results.append({
            'Model': model_name,
            'PSNR': 'N/A',
            'SSIM': 'N/A'
        })
        print(f"Skipping evaluation for {model_name} due to missing model.")
        continue

    total_psnr = 0
    total_ssim = 0
    num_samples = 0

    # Use the existing test_loader for evaluation
    with torch.no_grad():
        for lr_img, hr_img in test_loader:
            lr_img, hr_img = lr_img.to(device), hr_img.to(device)

            sr_img = model(lr_img).clamp(0, 1)

            hr_np = hr_img[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)
            sr_np = sr_img[0].permute(1, 2, 0).cpu().numpy().clip(0, 1)

            total_psnr += psnr_metric(hr_np, sr_np, data_range=1.0)
            total_ssim += ssim_metric(hr_np, sr_np, win_size=3, channel_axis=-1, data_range=1.0)
            num_samples += 1

    if num_samples > 0:
        avg_psnr = total_psnr / num_samples
        avg_ssim = total_ssim / num_samples
    else:
        avg_psnr = float('nan')
        avg_ssim = float('nan')

    results.append({
        'Model': model_name,
        'PSNR': round(avg_psnr, 4),
        'SSIM': round(avg_ssim, 4)
    })
    print(f"{model_name}: PSNR = {avg_psnr:.4f}, SSIM = {avg_ssim:.4f}")

# Display results in a DataFrame
ablation_df = pd.DataFrame(results)
print("\n" + "="*40)
print("Ablation Study Quantitative Results")
print("="*40)
display(ablation_df)

# Display individual ablation results as requested by the user
print("\n" + "="*40)
print("Individual Ablation Model Results")
print("="*40)

# Proposed - Baseline (No SC, No MIF)
baseline_row = ablation_df[ablation_df['Model'] == 'Proposed - Baseline (No SC, No MIF)']
if not baseline_row.empty:
    print("Proposed - Baseline (No SC, No MIF):")
    display(baseline_row)
else:
    print("Proposed - Baseline (No SC, No MIF) results not available.")

# Proposed - Spectral Conv Only
spec_only_row = ablation_df[ablation_df['Model'] == 'Proposed - Spectral Conv Only']
if not spec_only_row.empty:
    print("Proposed - Spectral Conv Only:")
    display(spec_only_row)
else:
    print("Proposed - Spectral Conv Only results not available.")

# Proposed - MIF Loss Only
mif_only_row = ablation_df[ablation_df['Model'] == 'Proposed - MIF Loss Only']
if not mif_only_row.empty:
    print("Proposed - MIF Loss Only:")
    display(mif_only_row)
else:
    print("Proposed - MIF Loss Only results not available.")

# Proposed (Full)
full_proposed_row = ablation_df[ablation_df['Model'] == 'Proposed (Full)']
if not full_proposed_row.empty:
    print("Proposed (Full):")
    display(full_proposed_row)
else:
    print("Proposed (Full) results not available.")

# Optionally save the ablation results to a CSV
ablation_results_path = os.path.join(metrics_dir, 'ablation_quantitative_results.csv')
ablation_df.to_csv(ablation_results_path, index=False)

In [ ]:
import pandas as pd
import os

# Ensure metrics_dir is defined (it should be from earlier cells)
# If running this cell independently, uncomment and define metrics_dir
# mpath = '/content/drive/MyDrive//'
# metrics_dir = mpath + 'models_metrics_messidor2'

CSV_MAP = {
    'VDSR'         : 'vdsr_metrics.csv',
    'EDSR'         : 'edsr_metrics.csv',
    'RCAN'         : 'rcan_metrics.csv',
    'Swin2SR'      : 'swin2sr_metrics.csv',
    'VisionMambaSR': 'mamba_metrics.csv',
    'DiffusionSR'  : 'diffusion_metrics.csv',
    'Proposed'     : 'proposed_messidor2_metrics.csv',
}

master_data = {}
for model, fname in CSV_MAP.items():
    fpath = os.path.join(metrics_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df.columns = [c.strip() for c in df.columns]
        master_data[model] = df
    else:
        print(f"  WARNING: {fpath} not found -- {model} skipped")

rows = []
metric_cols = ['PSNR', 'SSIM', 'MSE', 'MAE'] # Metrics to include in the final table

for model in CSV_MAP.keys():
    if model in master_data:
        df_model = master_data[model]
        # Filter out rows where any of the key metrics are NaN, then take the last one
        # If all rows are NaN for a specific metric, this will take the last available row
        # for other metrics, and then individual checks handle the N/A for display.
        last_valid_row = df_model.dropna(subset=metric_cols).iloc[-1] if not df_model.dropna(subset=metric_cols).empty else pd.Series(index=metric_cols, dtype=float)

        rows.append({
            'Model': model,
            'Scale': 4,
            'PSNR': round(float(last_valid_row['PSNR']), 3) if pd.notna(last_valid_row['PSNR']) else 'N/A',
            'SSIM': round(float(last_valid_row['SSIM']), 4) if pd.notna(last_valid_row['SSIM']) else 'N/A',
            'MSE': round(float(last_valid_row['MSE']), 5) if pd.notna(last_valid_row['MSE']) else 'N/A',
            'MAE': round(float(last_valid_row['MAE']), 5) if pd.notna(last_valid_row['MAE']) else 'N/A',
        })
    else:
        rows.append({'Model': model, 'Scale': 4, 'PSNR': 'N/A', 'SSIM': 'N/A', 'MSE': 'N/A', 'MAE': 'N/A'})

table_ii = pd.DataFrame(rows)
print("\n" + "="*68)
print("TABLE II -- Quantitative Comparison (Messidor-2, Scale x4)")
print("="*68)
display(table_ii)
print("="*68)

table_ii_path = os.path.join(metrics_dir, 'TABLE_II_messidor2_7models.csv')
table_ii.to_csv(table_ii_path, index=False)
print(f"\nTable saved to: {table_ii_path}")

In [ ]:
import pandas as pd
import os

# Ensure metrics_dir is defined from earlier cells
# mpath is defined in cell aefa06fe
metrics_dir = mpath + 'models_metrics_messidor2'

# Define model labels and their corresponding CSV filenames
ablation_file_map = {
    'Proposed (Full)': 'proposed_messidor2_metrics.csv', # Corrected filename
    'Proposed - No Spectral Conv': 'proposed_no_spectral_messidor2_metrics.csv',
    'Proposed - No MIF Loss': 'proposed_no_mif_messidor2_metrics.csv',
    # 'Proposed - Baseline (No SC, No MIF)': 'proposed_no_spectral_no_mif_messidor2_metrics.csv' # Excluded as per user request
}

metrics_to_extract = ['PSNR', 'SSIM', 'MSE', 'MAE']

ablation_results = []

for model_name, filename in ablation_file_map.items():
    fpath = os.path.join(metrics_dir, filename)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        df.columns = [c.strip() for c in df.columns] # Clean column names

        # Filter out rows with NaN in critical metrics before taking the last valid one
        last_valid_row = df.dropna(subset=metrics_to_extract).iloc[-1] if not df.dropna(subset=metrics_to_extract).empty else pd.Series(dtype=float)

        row_data = {'Model': model_name}
        for metric in metrics_to_extract:
            if metric in last_valid_row and pd.notna(last_valid_row[metric]):
                # Rounding based on typical precision for these metrics
                if metric == 'PSNR':
                    row_data[metric] = round(float(last_valid_row[metric]), 3)
                elif metric == 'SSIM':
                    row_data[metric] = round(float(last_valid_row[metric]), 4)
                elif metric == 'Entropy':
                    row_data[metric] = round(float(last_valid_row[metric]), 3)
                elif metric == 'Edge_Sim':
                    row_data[metric] = round(float(last_valid_row[metric]), 4)
                else: # MSE, MAE, RMSE, Blended_Loss
                    row_data[metric] = round(float(last_valid_row[metric]), 5)
            else:
                row_data[metric] = 'N/A'
        ablation_results.append(row_data)
    else:
        print(f"  WARNING: {fpath} not found -- {model_name} skipped.")
        row_data = {'Model': model_name}
        for metric in metrics_to_extract:
            row_data[metric] = 'N/A'
        ablation_results.append(row_data)

ablation_table = pd.DataFrame(ablation_results)

print("\n" + "="*80)
print("TABLE -- Ablation Study Quantitative Comparison (Messidor-2, Scale x4)")
print("="*80)
display(ablation_table)
print("="*80)

# Optionally save the ablation table to a CSV
ablation_table_path = os.path.join(metrics_dir, 'TABLE_Ablation_messidor2.csv')
ablation_table.to_csv(ablation_table_path, index=False)
print(f"\nTable saved to: {ablation_table_path}")

### Ablation Study Results: PSNR and SSIM Comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better aesthetics
sns.set_style("whitegrid")

# Create a figure and a primary axes for PSNR
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot PSNR as a line on the primary y-axis
sns.lineplot(x='Model', y='PSNR', data=ablation_df, marker='o', linestyle='-', color='darkgreen', label='PSNR', ax=ax1)
ax1.set_title('PSNR and SSIM Comparison for Ablation Study Models', fontsize=16)
ax1.set_xlabel('Model Configuration', fontsize=12)
ax1.set_ylabel('PSNR (dB)', fontsize=12, color='darkgreen')
ax1.tick_params(axis='y', labelcolor='darkgreen')
# Adjust y-axis limits to prevent lines from being cut off
ax1.set_ylim(ablation_df['PSNR'].min() * 0.95, ablation_df['PSNR'].max() * 1.05)

# Create a secondary y-axis for SSIM
ax2 = ax1.twinx()
# Plot SSIM as a line on the secondary y-axis
sns.lineplot(x='Model', y='SSIM', data=ablation_df, marker='s', linestyle='--', color='darkred', label='SSIM', ax=ax2)
ax2.set_ylabel('SSIM', fontsize=12, color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')
# Adjust y-axis limits to prevent lines from being cut off
ax2.set_ylim(ablation_df['SSIM'].min() * 0.95, ablation_df['SSIM'].max() * 1.05)

# Add legends. For twinx plots, it's common to collect handles and labels from both axes
# and create a single legend, using parameters from the example for compactness.
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left', ncol=2, fontsize=10, framealpha=0.9, edgecolor='gray')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Ensure thop is installed for FLOPs calculation, though not strictly needed for just time
try:
    from thop import profile, clever_format
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'thop', '-q'])
    from thop import profile, clever_format

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Dummy input (standard LR size for scale x4) ---
dummy_lr = torch.randn(1, 3, 64, 64).to(device)

# --- Using model instances defined and loaded in earlier cells (for Messidor-2) ---
# Ensure these models have been trained and are available in the current session.
models_dict = {
    'VDSR'          : vdsr_model,
    'EDSR'          : edsr_model,
    'RCAN'          : rcan_model,
    'Swin2SR'       : swin_model,
    'VisionMambaSR' : mamba_model,
    'DiffusionSR'   : diffusion_model,
    'Proposed (Full)': proposed_full_model,
}

# --- Latency Benchmarking ---
N_RUNS = 100 # Number of runs for stable timing
WARMUP = 20  # Warmup runs to allow GPU to allocate resources

latency_results = []

print("\n--- Inference Latency Benchmarking ---")
for name, model in models_dict.items():
    model.eval()
    print(f"Benchmarking: {name.replace(chr(10), ' ')}...")

    # Warmup runs
    with torch.no_grad():
        for _ in range(WARMUP):
            try:
                if 'Diffusion' in name: # Diffusion models use a 'sample' method
                    _ = model.sample(dummy_lr, steps=10) # Using 10 steps for faster inference
                else:
                    _ = model(dummy_lr)
            except Exception as e:
                print(f"  Warmup error for {name}: {e}")
                pass

    # Synchronize GPU before actual timing
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Timed runs
    times = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            t_start = time.perf_counter()
            try:
                if 'Diffusion' in name:
                    _ = model.sample(dummy_lr, steps=10) # 10 steps for consistency
                else:
                    _ = model(dummy_lr)
            except Exception as e:
                # Capture potential errors during timed runs too
                print(f"  Timed run error for {name}: {e}")
                continue # Skip this run if an error occurs

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()
            times.append((t_end - t_start) * 1000) # Convert to milliseconds

    if times: # Ensure there are valid times to calculate
        avg_time = np.mean(times)
        std_time = np.std(times)
    else:
        avg_time = np.nan
        std_time = np.nan

    print(f"  Average Inference Time: {avg_time:.3f} \u00B1 {std_time:.3f} ms")

    latency_results.append({
        'Model': name.replace('\n', ' '),
        'Average Latency (ms)': round(avg_time, 3),
        'Std Dev Latency (ms)': round(std_time, 3),
    })

# --- Display Results Table ---
df_latency = pd.DataFrame(latency_results)
print("\n" + "="*70)
print("TABLE IV \u2014 Inference Latency Analysis (Single Image, 1x3x64x64 input)")
print("="*70)
print(df_latency.to_string(index=False))
print("="*70)

# --- Save Table to CSV ---
# The 'metrics_dir' is already defined for Messidor-2 metrics
csv_path = os.path.join(metrics_dir, 'Table4_latency.csv')
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df_latency.to_csv(csv_path, index=False)
print(f"\nLatency results saved to: {csv_path}")

# --- Plotting Latency ---
model_names_plot = df_latency['Model'].tolist()
latency_values = df_latency['Average Latency (ms)'].tolist()
std_dev_values = df_latency['Std Dev Latency (ms)'].tolist()

plt.figure(figsize=(10, 6), dpi=150)
bars = plt.bar(model_names_plot, latency_values, yerr=std_dev_values, capsize=5,
               color=plt.cm.viridis(np.linspace(0, 1, len(model_names_plot))),
               edgecolor='black', linewidth=0.7)

plt.xlabel("Model", fontsize=12)
plt.ylabel("Average Latency (ms)", fontsize=12)
plt.title("Fig. 7. Average Inference Latency per Image (Lower is Better)", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.bar_label(bars, fmt='%.2f', padding=5, fontsize=9)
plt.tight_layout()

# The 'results_dir' is already defined for Messidor-2 results
chart_path = os.path.join(results_dir, 'plots', 'Fig7_Latency_Benchmark.png')
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
plt.savefig(chart_path, bbox_inches='tight', dpi=200)
plt.show()
plt.close()
print(f"\nLatency chart saved to: {chart_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure the output directory exists
# Correcting path to reflect Scale x4
os.makedirs(plots_dir + 'Ablation_plots//4//', exist_ok=True)

# Define the base path to the metrics CSV files
metrics_base_path = mpath + 'models_metrics_messidor2//'

# Define a mapping from model labels to their respective CSV filenames
# Exclude 'Proposed - Baseline (No SC, No MIF)' as per user request
file_name_map = {
    'MIF + Spec (Full Model)': 'ablation_Full_Proposed.csv',
    'MIF + No-Spec': 'proposed_no_spectral_messidor2_metrics.csv', # This corresponds to use_spectral=False, MIF_Loss=True
    'No-MIF + Spec': 'proposed_no_mif_messidor2_metrics.csv',         # This corresponds to use_spectral=True, MIF_Loss=False
}

# Metrics to plot - updated to reflect actual metrics saved in CSVs
metrics_to_plot = ['PSNR', 'SSIM', 'MSE', 'MAE', 'RMSE', 'Entropy', 'Edge_Sim', 'Blended_Loss']

# Define the maximum number of epochs to plot for consistency across ablation models.
# Proposed (Full) was trained for 50 epochs, while ablation models were trained for 25.
# Plotting up to 25 epochs ensures a fair comparison across all. If you want to see
# the full 50 epochs for the Proposed (Full) model, you would need to adjust this.
max_epoch_for_plots = 25

# Custom color palette for the plots
custom_palette = sns.color_palette(["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"])

# Loop through each metric to create and save plots
for metric in metrics_to_plot:
    plt.figure(figsize=(10, 6))
    sns.set_palette(custom_palette)

    for label, filename in file_name_map.items():
        df_path = os.path.join(metrics_base_path, filename)
        if os.path.exists(df_path):
            df = pd.read_csv(df_path)
            df.columns = [c.strip() for c in df.columns] # Clean column names

            # Filter to the consistent epoch range
            df = df[df['Epoch'] <= max_epoch_for_plots]

            # Convert metric columns to numeric, coercing errors to NaN
            if metric in df.columns:
                df[metric] = pd.to_numeric(df[metric], errors='coerce')

            # Plot if the metric column exists and has non-NaN values
            if metric in df.columns and not df[metric].isnull().all():
                sns.lineplot(x='Epoch', y=metric, data=df, label=label)
            else:
                print(f"Warning: Metric '{metric}' not found or all NaN in {filename}. Skipping plot for this metric and file.")
        else:
            print(f"Warning: File not found: {df_path}. Skipping plot for this file.")

    plt.title(f'{metric} over Epochs for Ablation Models (Scale x4)') # Corrected title
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.grid(True)
    plt.legend(title='Model')
    plt.tight_layout()
    # Corrected filename
    plt.savefig(os.path.join(plots_dir + 'Ablation_plots//4//', f'ablation_scale4_{metric}_filtered.png'))
    plt.show()

print("Ablation plots generated and saved.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure the output directory exists
os.makedirs(plots_dir + 'Ablation_plots//4//', exist_ok=True)

# Define the base path to the metrics CSV files
metrics_base_path = mpath + 'models_metrics_messidor2//'

# Define a mapping from model labels to their respective CSV filenames
# Using shorter aliases for clearer plot legends
file_name_map = {
    'Proposed (Full)': 'ablation_Full_Proposed.csv',
    'Proposed (No SC)': 'proposed_no_spectral_messidor2_metrics.csv',
    'Proposed (No MIF)': 'proposed_no_mif_messidor2_metrics.csv',
}

# Define the maximum number of epochs to plot for consistency across ablation models.
max_epoch_for_plots = 25

# Explicitly define order, colors, markers, and linestyles for clarity
ABLATION_MODEL_ORDER = [
    'Proposed (No SC)',
    'Proposed (No MIF)',
    'Proposed (Full)'
]
ABLATION_COLORS = {
    'Proposed (Full)': '#1f77b4',       # Blue
    'Proposed (No SC)': '#ff7f0e',      # Orange
    'Proposed (No MIF)': '#2ca02c',     # Green
}

ABLATION_LINESTYLES_PSNR = {
    'Proposed (Full)': '-',
    'Proposed (No SC)': '-',
    'Proposed (No MIF)': '-',
}
ABLATION_LINESTYLES_SSIM = {
    'Proposed (Full)': '--',
    'Proposed (No SC)': '--',
    'Proposed (No MIF)': '--',
}

# Load data for the selected models
all_models_df = pd.DataFrame()
for label, filename in file_name_map.items():
    df_path = os.path.join(metrics_base_path, filename)
    if os.path.exists(df_path):
        df = pd.read_csv(df_path)
        df.columns = [c.strip() for c in df.columns]  # Clean column names
        df['Model'] = label  # Assign the shorter label
        # Filter to the consistent epoch range
        df = df[df['Epoch'] <= max_epoch_for_plots]
        all_models_df = pd.concat([all_models_df, df], ignore_index=True)
    else:
        print(f"Warning: File not found: {df_path}. Skipping data for {label}.")

if not all_models_df.empty:
    plt.figure(figsize=(12, 7))

    # Initialize primary y-axis for PSNR
    ax1 = plt.gca()
    ax1.set_ylabel('PSNR (dB)', fontsize=12, color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_title('PSNR and SSIM over Epochs for Ablation Models (Scale x4)', fontsize=14)

    # Initialize secondary y-axis for SSIM
    ax2 = ax1.twinx()
    ax2.set_ylabel('SSIM', fontsize=12, color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    # Plot PSNR and SSIM for each model explicitly
    for model_label in ABLATION_MODEL_ORDER:
        model_df = all_models_df[all_models_df['Model'] == model_label]
        if not model_df.empty:
            # Plot PSNR line
            sns.lineplot(x='Epoch', y='PSNR', data=model_df,
                         color=ABLATION_COLORS[model_label],
                         linestyle=ABLATION_LINESTYLES_PSNR[model_label],
                         label=f'{model_label} (PSNR)',
                         ax=ax1, linewidth=3)
            # Plot SSIM line
            sns.lineplot(x='Epoch', y='SSIM', data=model_df,
                         color=ABLATION_COLORS[model_label],
                         linestyle=ABLATION_LINESTYLES_SSIM[model_label],
                         label=f'{model_label} (SSIM)',
                         ax=ax2, linewidth=3)

    # Combine legends from both axes
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()

    # Remove the individual auto-generated legends on each axis
    if ax1.get_legend():
        ax1.get_legend().remove()
    if ax2.get_legend():
        ax2.get_legend().remove()

    # Place combined legend inside the plot, bottom-right corner
    ax1.legend(handles1 + handles2, labels1 + labels2,
               loc='lower right',
               bbox_to_anchor=(0.98, 0.02),
               ncol=1,
               frameon=True,
               facecolor='white',
               framealpha=0.85,
               edgecolor='gray',
               fontsize=9)

    ax1.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    combined_plot_path = os.path.join(plots_dir + 'Ablation_plots//4//', 'ablation_scale4_PSNR_SSIM_combined.png')
    plt.savefig(combined_plot_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Combined PSNR and SSIM plot generated and saved to: {combined_plot_path}")
else:
    print("No data available to generate the combined PSNR and SSIM plot.")

In [ ]:
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Install thop for FLOPs calculation
try:
    from thop import profile, clever_format
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'thop', '-q'])
    from thop import profile, clever_format

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Dummy input (standard LR size for scale x4) ───────────────
# LR input = 64x64, HR output = 256x256
dummy_lr = torch.randn(1, 3, 64, 64).to(device)

# ── All models with their display names ───────────────────────
# Using model instances defined and loaded in earlier cells (for Messidor-2)
models_dict = {
    'VDSR'          : vdsr_model,
    'EDSR'          : edsr_model,
    'RCAN'          : rcan_model,
    'Swin2SR'       : swin_model,
    'VisionMambaSR' : mamba_model,
    'DiffusionSR'   : diffusion_model,
    'Proposed (Full)' : proposed_full_model,
}

# ── Measure FLOPs, Params, and Execution Time ─────────────────
N_RUNS    = 50    # number of runs for stable timing
WARMUP    = 10    # warmup runs before timing

results = []

for name, model in models_dict.items():
    model.eval()
    print(f"\nAnalysing: {name.replace(chr(10), ' ')} ...")

    # ── 1. Parameters ─────────────────────────────────────────
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters()
                           if p.requires_grad)

    # ── 2. FLOPs ──────────────────────────────────────────────
    try:
        # thop needs CPU model for some architectures
        model_cpu  = model.cpu()
        dummy_cpu  = dummy_lr.cpu()
        flops, _   = profile(model_cpu, inputs=(dummy_cpu,),
                             verbose=False)
        model.to(device)
    except Exception:
        # Fallback for models with custom forward (Diffusion)
        try:
            # For DiffusionSR count only UNet FLOPs
            unet       = model.unet.cpu()
            cond_cpu   = dummy_cpu
            x_cpu      = torch.randn(1, 3, 256, 256)  # HR noise
            t_cpu      = torch.zeros(1, dtype=torch.long)
            flops, _   = profile(unet,
                                 inputs=(x_cpu, cond_cpu, t_cpu),
                                 verbose=False)
            model.to(device)
            unet.to(device)
        except Exception as e:
            print(f"  FLOPs calculation skipped: {e}")
            flops = 0
            model.to(device)

    # ── 3. Execution Time ─────────────────────────────────────
    dummy_lr = dummy_lr.to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(WARMUP):
            try:
                if 'Diffusion' in name:
                    _ = model.sample(dummy_lr, steps=10)
                else:
                    _ = model(dummy_lr)
            except Exception:
                pass

    # Synchronize GPU before timing
    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Timed runs
    times = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            t_start = time.perf_counter()
            try:
                if 'Diffusion' in name:
                    out = model.sample(dummy_lr, steps=10)
                else:
                    out = model(dummy_lr)
            except Exception:
                pass
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()
            times.append((t_end - t_start) * 1000)  # ms

    avg_time = np.mean(times)
    std_time = np.std(times)

    # ── Format values ─────────────────────────────────────────
    flops_m  = flops / 1e6  if flops > 0 else 0
    params_k = total_params / 1e3

    print(f"  FLOPs      : {flops_m:.3f} M")
    print(f"  Params     : {params_k:.3f} K  "
          f"(trainable: {trainable_params/1e3:.3f} K)")
    print(f"  Exec Time  : {avg_time:.2f} \u00B1 {std_time:.2f} ms")

    results.append({
        'Model'           : name.replace('\n', ' '),
        'FLOPs (M)'       : round(flops_m,         3),
        'Params (K)'      : round(params_k,         3),
        'Trainable (K)'   : round(trainable_params/1e3, 3),
        'Exec Time (ms)'  : round(avg_time,         2),
        'Std Time (ms)'   : round(std_time,          2),
    })

# ── Print Table ───────────────────────────────────────────────
df = pd.DataFrame(results)
print("\n" + "="*75)
print("TABLE III — Model Complexity & Execution Time Analysis")
print("="*75)
print(df.to_string(index=False))
print("="*75)

# Save table to CSV
csv_path = os.path.join(metrics_dir, 'Table3_complexity.csv')
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"\nTable saved to: {csv_path}")


# ── Bar Charts ────────────────────────────────────────────────
model_names  = [r['Model'] for r in results]
flops_vals   = [r['FLOPs (M)']      for r in results]
params_vals  = [r['Params (K)']     for r in results]
time_vals    = [r['Exec Time (ms)'] for r in results]
std_vals     = [r['Std Time (ms)']  for r in results]

COLORS = ['#4878d0','#ee854a','#6acc65','#d65f5f',
          '#956cb4','#8c613c','#dc7ec0']

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=130)

# ── Plot 1: FLOPs ─────────────────────────────────────────────
bars1 = axes[0].bar(model_names, flops_vals,
                    color=COLORS, edgecolor='black', linewidth=0.7)
axes[0].set_title("FLOPs (M) — Lower is Better",
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel("FLOPs (M)", fontsize=10)
axes[0].tick_params(axis='x', rotation=30, labelsize=8)
axes[0].bar_label(bars1, fmt='%.2f', fontsize=8, padding=2)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# ── Plot 2: Parameters ────────────────────────────────────────
bars2 = axes[1].bar(model_names, params_vals,
                    color=COLORS, edgecolor='black', linewidth=0.7)
axes[1].set_title("Parameters (K) — Lower is Better",
                  fontsize=11, fontweight='bold')
axes[1].set_ylabel("Params (K)", fontsize=10)
axes[1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1].bar_label(bars2, fmt='%.1f', fontsize=8, padding=2)
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# ── Plot 3: Execution Time ────────────────────────────────────
bars3 = axes[2].bar(model_names, time_vals,
                    color=COLORS, edgecolor='black', linewidth=0.7,
                    yerr=std_vals, capsize=4, error_kw={'linewidth':1.2})
axes[2].set_title("Execution Time (ms) — Lower is Better",
                  fontsize=11, fontweight='bold')
axes[2].set_ylabel("Time (ms)", fontsize=10)
axes[2].tick_params(axis='x', rotation=30, labelsize=8)
axes[2].bar_label(bars3, fmt='%.1f', fontsize=8, padding=2)
axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle(
    "Table III — Quantitative Analysis of Model Complexity\n"
    "FLOPs | Parameters | Execution Time  (Input: 1\u00d73\u00d764\u00d764, Scale \u00d74)",
    fontsize=11, fontweight='bold'
)
plt.tight_layout()

chart_path = os.path.join(results_dir, 'plots', 'Table3_Complexity_Chart.png')
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
plt.savefig(chart_path, bbox_inches='tight', dpi=220)
plt.show()
plt.close()
print(f"Chart saved to: {chart_path}")